# 05 · Gold: a trust score for every facility × service

**Question this answers:** for any service a patient needs (dialysis, NICU, cataract surgery, ... 2,531 services), which facilities can we *trust* to provide it, and what is the evidence?

**Input:** workspace.default.silver_facilities (9,997 cleaned facilities, built in 03_silver_master).

**Output tables**

| table | what it is |
|---|---|
| gold_facility_services | one row per facility × service: trust_level (Verified / Likely / Claimed only), trust_score 0-100, quoted evidence, score_reasons, problems (cautions), explanation (one plain-language sentence for users, with the cautions), new flags (flag_ayush_claim, flag_dubious_claim, flag_other_facility) |
| gold_services | catalogue: one row per service with the number of facilities at each trust level |
| gold_facility_services_v1, gold_services_v1 | the first version, kept for the before/after comparison |
| verify_sample_v1/v2/v3, verify_results_v1/v2/v3 | the rows we checked by hand and our verdicts |

**How one facility × service is scored (Step 3)**
- Evidence: procedure/capability mentions it +30, equipment +25, description +15, listed specialty +10, concrete numbers (beds, machines) +10, stated in several separate places +10, large hospital (200+ beds) +10, 24/7 wording +5 (emergency services), size known +5.
- Penalties: dentist/pharmacy claiming hospital care −40, eye/dental/ENT/skin specialist (named as such) claiming emergency or maternity care −40 (fix I), homeopathy/ayurveda/unani facility claiming any modern-medicine service −30, record merges two places −30, pseudo-science wording −20, evidence names a similarly-named other hospital −20, clinic without beds claiming hospital care −15, address state conflicts with PIN −15, duplicate / inflated specialty list / unrealistic counts −10, coordinates far from PIN −5.
- **Verified**: score ≥ 60, at least 2 independent kinds of support, no serious problem.
- **Likely**: score ≥ 35 and at least one hard piece of evidence.
- **Claimed only**: only a specialty tag or weak text. Shown with a warning.

**Run order:** Steps 1 → 15, top to bottom. Run all is safe: the verification samples are stored tables and are never re-drawn. The Appendix is the old v1 code and does nothing when run.

**How we know it works (Steps 6-15)**

| check | result |
|---|---|
| Round 1: 100 rows from v1, checked by hand | Verified 30/40 correct, 3 wrong. The 9 errors were used to design fixes A-G |
| Two reviewers on the same 10 rows | agreed on 9/10 (never Yes vs No) |
| Round 2: 30 unseen rows from v2 (the fair test) | Verified **11/12 correct, 0 wrong** |
| Round 3: 20 rows that became Verified only because of fix H | **15/20 correct, 0 wrong** (4 unsure, 1 not a service) |
| Famous-hospital test (Step 11) | Kidwai: cancer treatment Verified, #2 in Bengaluru. Tata Memorial: chemotherapy Verified, #9 in Mumbai (was #25 before fix H) |
| Fix I (Nov 21, from the app check: an eye hospital showed up as a 'Likely' emergency C-section) | 542 rows penalised (534 now Claimed only, 8 still Likely); Verified 35,778 → 35,753. Hand-check results and all 06_analysis numbers unchanged |

**Known limits:** the system can only recommend facilities that are in the given data; evidence that really belongs to another branch of a hospital chain cannot always be told apart from the text; records with almost no text (e.g. PGIMER Chandigarh) stay low.

In [0]:
# Step 1: inventory of ALL specialties in the data (how many distinct services do we really have?)
import re, json
import numpy as np
import pandas as pd

silver = spark.table("workspace.default.silver_facilities").toPandas()
as_list = lambda x: [str(i) for i in x] if x is not None and not (isinstance(x, float) and np.isnan(x)) else []

def norm_code(s):
    """'orthopedicSurgery' / 'Orthopedic surgery' / 'orthopedic-surgery' -> 'orthopedic surgery'"""
    s = re.sub(r"([a-z])([A-Z])", r"\1 \2", str(s))          # split camelCase
    s = re.sub(r"[^a-zA-Z]+", " ", s).strip().lower()
    return re.sub(r"\s+", " ", s)

fac_codes = silver["specialties"].apply(lambda xs: sorted({norm_code(x) for x in as_list(xs) if norm_code(x)}))
counts = fac_codes.explode().value_counts()
total_entries = counts.sum()
print(f"distinct raw specialty strings -> {len(counts):,} distinct normalised specialties, {total_entries:,} facility-specialty pairs")
for k in [1, 2, 5, 10, 20, 50]:
    sub = counts[counts >= k]
    print(f"  in >= {k:>2} facilities: {len(sub):>5} specialties covering {sub.sum() / total_entries:.1%} of all pairs")
print("\nTop 120 specialties (facilities):")
print("; ".join(f"{c} ({n})" for c, n in counts.head(120).items()))
print("\nExamples seen in only 1-2 facilities:")
print("; ".join(counts[counts <= 2].sample(40, random_state=1).index))

distinct raw specialty strings -> 2,530 distinct normalised specialties, 117,967 facility-specialty pairs
  in >=  1 facilities:  2530 specialties covering 100.0% of all pairs
  in >=  2 facilities:   903 specialties covering 98.6% of all pairs
  in >=  5 facilities:   449 specialties covering 97.6% of all pairs
  in >= 10 facilities:   299 specialties covering 96.8% of all pairs
  in >= 20 facilities:   221 specialties covering 95.9% of all pairs
  in >= 50 facilities:   152 specialties covering 94.1% of all pairs

Top 120 specialties (facilities):
internal medicine (6836); family medicine (5176); gynecology and obstetrics (4489); dentistry (4193); orthopedic surgery (3520); pediatrics (3429); general surgery (3195); cardiology (3007); radiology (2998); ophthalmology (2826); otolaryngology (2699); urology (2628); gastroenterology (2422); dermatology (2331); pathology (2252); nephrology (2156); emergency medicine (2081); neurology (1963); reproductive endocrinology and infertility (184

In [0]:
# Step 2: define ALL services = 10 hand-curated services + EVERY specialty that appears in the data
# Matching is done on normalised text: lower case, British spellings folded (paediatric -> pediatric, haem -> hem).
STOP = {"and", "or", "of", "the", "in", "with", "for", "a", "an", "to"}
nz = lambda s: re.sub(r"ae|oe", "e", str(s).lower())

def stem(w):
    w = nz(w)
    return w if len(w) <= 6 else w[:max(5, len(w) - 3)]

# --- 10 hand-curated services (richer keywords + equipment; REVIEW: medical judgement) ---
CURATED = {
    "dialysis": dict(label="Dialysis", hospital_level=False, emergency=False,
        specialty=r"nephrolog", text=r"dialysis|hemodialysis|renal replacement|\bcapd\b|kidney care unit",
        equipment=r"dialys|dialyz|\bro plant\b"),
    "emergency_c_section": dict(label="Emergency C-section", hospital_level=True, emergency=True,
        specialty=r"obstetric|maternalfetal", text=r"c-?section|cesarean|\blscs\b",
        equipment=r"operation theat|operating theat|\bot\b|labou?r (room|ward)|delivery (room|suite)"),
    "nicu": dict(label="Newborn intensive care (NICU)", hospital_level=True, emergency=True,
        specialty=r"neonatolog|perinatal", text=r"\bnicu\b|neonatal intensive|newborn intensive|\bsncu\b|level (ii|iii|2|3) neonatal",
        equipment=r"incubator|radiant warmer|phototherapy|neonatal ventilator|bubble cpap"),
    "cardiac_surgery": dict(label="Cardiac surgery", hospital_level=True, emergency=False,
        specialty=r"cardiacsurg|cardiothoracic|cardiovascularsurg",
        text=r"cardiac surgery|heart surgery|open[- ]heart|bypass surgery|\bcabg\b|valve (replacement|repair)|cardiothoracic",
        equipment=r"heart[- ]lung machine|cardiopulmonary bypass|cath ?lab"),
    "mri": dict(label="MRI scan", hospital_level=False, emergency=False,
        specialty=r"radiolog", text=r"\bmri\b|magnetic resonance", equipment=r"\bmri\b|magnetic resonance"),
    "ct_scan": dict(label="CT scan", hospital_level=False, emergency=False,
        specialty=r"radiolog", text=r"\bct\b|ct[- ]scan|computed tomography|\bcat scan\b",
        equipment=r"\bct\b|ct[- ]scan|computed tomography|slice ct"),
    "burn_care": dict(label="Burn care", hospital_level=True, emergency=True,
        specialty=r"burn", text=r"\bburns? (unit|ward|care|centre|center|icu|treatment|management|surgery)|burn injur|treatment of burns",
        equipment=r"burn"),
    "trauma_emergency": dict(label="Trauma / emergency care", hospital_level=True, emergency=True,
        specialty=r"emergencymedicine|traumasurg|traumatolog",
        text=r"\btrauma (care|centre|center|unit|surgery)|emergency (department|room|ward|services?|care)|\bcasualty\b|accident",
        equipment=r"defibrillator|ambulance|ventilator|crash cart"),
    "icu": dict(label="Intensive care (ICU)", hospital_level=True, emergency=True,
        specialty=r"criticalcare|intensivecare", text=r"\bicu\b|intensive care|critical care|\biccu\b|\bmicu\b|\bsicu\b|\bccu\b",
        equipment=r"ventilator|icu bed|multipara(meter)? monitor|central monitoring"),
    "cancer_treatment": dict(label="Cancer treatment (chemo / radiation)", hospital_level=False, emergency=False,
        specialty=r"oncolog", text=r"chemotherap|radiotherap|radiation (therapy|oncology)|cancer (treatment|care|surgery|centre|center)|oncology",
        equipment=r"linear accelerator|\blinac\b|brachytherap|cobalt|tomotherapy|pet[- ]ct"),
}

# --- automatic service for every normalised specialty in the data ---
HOSP_WORDS = ["transplant", "intensive", "critical", "neonat", "trauma", "cardiac surg", "cardiothoracic", "neurosurg", "burn"]
EMERG_WORDS = ["emergency", "critical", "trauma", "intensive", "neonat", "burn"]

def auto_service(code):
    words = [w for w in code.split() if w not in STOP] or code.split()
    stems = [stem(w) for w in words]
    # short words must match exactly (so 'ent' does not match 'entrance'); longer stems match any ending
    parts = [re.escape(s) + (r"\b" if len(s) <= 4 else r"\w*") for s in stems]
    pattern = r"\b" + r"\W+(?:\w+\W+){0,2}?".join(parts)      # words in order, up to 2 words in between
    return dict(label=code.title(), code=code, stems=stems, source="specialty",
                hospital_level=any(k in code for k in HOSP_WORDS),
                emergency=any(k in code for k in EMERG_WORDS),
                specialty=None, text=pattern, equipment=pattern)

SERVICES = {k: dict(s, source="curated", code=None, stems=None) for k, s in CURATED.items()}
for code in counts.index:                       # all 2,530 normalised specialties
    SERVICES["sp_" + code.replace(" ", "_")] = auto_service(code)
for s in SERVICES.values():
    s["text_re"] = re.compile(s["text"], re.I)
    s["equipment_re"] = re.compile(s["equipment"], re.I)
    s["specialty_re"] = re.compile(s["specialty"], re.I) if s["specialty"] else None

n_cur = sum(s["source"] == "curated" for s in SERVICES.values())
print(f"{len(SERVICES):,} services: {n_cur} curated + {len(SERVICES) - n_cur:,} from specialties")
print("example auto patterns:", {k: SERVICES[k]["text"] for k in ["sp_nephrology", "sp_pediatric_surgery", "sp_cataract_and_anterior_segment_surgery"] if k in SERVICES})

2,540 services: 10 curated + 2,530 from specialties
example auto patterns: {'sp_nephrology': '\\bnephrol\\w*', 'sp_pediatric_surgery': '\\bpediat\\w*\\W+(?:\\w+\\W+){0,2}?surge\\w*', 'sp_cataract_and_anterior_segment_surgery': '\\bcatar\\w*\\W+(?:\\w+\\W+){0,2}?anter\\w*\\W+(?:\\w+\\W+){0,2}?segme\\w*\\W+(?:\\w+\\W+){0,2}?surge\\w*'}


In [0]:
# Step 3: GOLD v2 = Gold v1 + fixes found by three rounds of manual verification (Joyce, Tim, Daniel)
# Needs Steps 1-2 (silver, norm_code, counts, nz, stem, STOP, CURATED ...). Writes gold_facility_services and gold_services.
#   A. drop "services" that are academic subjects or facility descriptions, not patient care
#      (Physiology, Anatomy ... sample row 1; "Multispeciality" ... round-3 row 210)
#   B. a multi-word specialty must stay inside one list item: words may be up to 2 apart, never across , ; .   row 14
#      (a trailing field word is dropped: "Glaucoma Ophthalmology" is found by "glaucoma")                 row 86
#   C. curated services ignore imaging/lab specialty tags ("cardiothoracic radiology" is not surgery)   row 63
#   D. pseudo-science wording ("energy quantum", reiki ...) is not evidence, and is flagged         row 96
#   E. homeopathy / ayurveda / unani facilities claiming ANY modern-medicine service: -30            row 47
#   G. (from round 2) right keyword, wrong context: "Cancer treatment (chemo / radiation)" needs chemo/radiation, not
#      cancer surgery; an IVF-lab incubator is not a NICU; an eye/dental emergency is not general emergency care   rows 108, 117, 128
#   F. evidence that names a similarly-named but DIFFERENT hospital: -20 and can not be Verified
#   H. (from the famous-hospital test) big hospitals list everything in ONE field, so they never reached Verified:
#      several separate statements: +10 and counts as a 2nd kind of evidence; concrete numbers count as a 2nd kind too;
#      a large hospital (200+ beds, counts not flagged) gets +10 for a curated / hospital-level service it has evidence for                        row 20
import copy, datetime, difflib
from bisect import bisect_left
from collections import defaultdict

# 0) keep v1 for the before/after comparison (first run only)
for t in ["gold_facility_services", "gold_services"]:
    if not spark.catalog.tableExists(f"workspace.default.{t}_v1"):
        spark.sql(f"CREATE TABLE workspace.default.{t}_v1 AS SELECT * FROM workspace.default.{t}")

# A) academic subjects that are not a service a patient can receive
NOT_A_SERVICE = re.compile(r"^(physiology|anatomy|human anatomy|pharmacology|forensic medicine( and toxicology)?|"
                           r"forensic medicine toxicology|community medicine|preventive and social medicine|"
                           r"social and preventive medicine|medical education|research|independent divisions|"
                           r"basic sciences?|administration)$")
# A2 (from round 3): words that describe the FACILITY, not a service a patient can ask for ("Multispeciality")
FACILITY_WORDS = re.compile(r"^(multi ?speciality|multi ?specialty|multi ?specialities|multi ?specialties|super ?speciality|"
                            r"super ?specialty|multi ?speciality hospital|super ?speciality hospital|general hospital|hospital|"
                            r"hospitals|clinic|clinics|polyclinic|nursing home|healthcare|health care|speciality|specialty|"
                            r"specialities|specialties|all specialities|all specialties|others?)$")
_na = NOT_A_SERVICE
NOT_A_SERVICE = re.compile(f"(?:{_na.pattern})|(?:{FACILITY_WORDS.pattern})")
print("A) removed, not a service:", [(c, int(counts[c])) for c in counts.index if NOT_A_SERVICE.match(c)])

# B) phrase patterns: words in order, up to 2 words in between, but never across a comma, semicolon or full stop
#    (v1 allowed any punctuation, so "immunoassays, haematology" matched "Immuno Haematology")
GAP = r"[ \t\-/&]+(?:[a-z]+[ \t\-/&]+){0,2}?"
GAP_STOP = GAP
def phrase_pattern(code):
    req, gaps, saw_stop = [], [], False
    for w in code.split():
        if w in STOP:
            saw_stop = bool(req)
            continue
        s = stem(w)
        req.append(re.escape(s) + (r"\b" if len(s) <= 4 else r"\w*"))
        gaps.append(saw_stop)
        saw_stop = False
    if not req:
        return r"\b" + re.escape(nz(code)) + r"\b"
    pat = r"\b" + req[0]
    for p, g in zip(req[1:], gaps[1:]):
        pat += (GAP_STOP if g else GAP) + p
    return pat
if "_auto_service_v1" not in globals():
    _auto_service_v1 = auto_service
# a trailing field word adds nothing to the search: "glaucoma ophthalmology" -> look for "glaucoma"
FIELD_SUFFIX = re.compile(r"\s+(ophthalmology|otolaryngology|pmr|internal medicine)$")
def auto_service_v2(code):
    s = _auto_service_v1(code)
    core = FIELD_SUFFIX.sub("", code) if len(code.split()) > 1 else code
    s["stems"] = [stem(w) for w in core.split() if w not in STOP] or [stem(w) for w in core.split()]
    s["text"] = s["equipment"] = phrase_pattern(core)
    return s

# C) curated services: imaging/lab specialty tags do not count (a cath lab stays: removing it lost correct hospitals)
CURATED2 = copy.deepcopy(CURATED)
# G) the label promises chemo / radiation, so only chemo / radiation evidence counts (cancer SURGERY or a "cancer centre" does not)
CURATED2["cancer_treatment"]["text"] = r"chemotherap|\bchemo\b|radiotherap|radiation (therapy|oncology)|medical oncolog|radiation oncolog|hemato[- ]?oncolog"
CURATED2["cancer_treatment"]["specialty"] = r"medicaloncolog|radiationoncolog|hematooncolog|hematologyoncolog|oncologyhematolog|clinicaloncolog"
CURATED2["cancer_treatment"]["equipment"] = r"linear accelerator|\blinac\b|brachytherap|cobalt|tomotherapy"
# G) a line that mentions the keyword in another context does not count for these services
EXCLUDE = {"nicu": re.compile(r"embryo|\bivf\b|in[- ]vitro|\blab\b|laborator|culture|andrology|semen|blastocyst", re.I),
           "trauma_emergency": re.compile(r"ophthalm|\beyes?\b|ocular|retina|dental|\btooth|teeth|veterinar", re.I)}
# I) (from the app check: an eye hospital showed up as a "Likely" emergency C-section) a facility whose NAME says it is an
#    eye / dental / ENT / skin / physiotherapy / veterinary specialist is not an emergency or maternity hospital: -40, never Verified.
#    Names that also say maternity / women / child / general / multi / nursing home / medical college are not touched.
SPECIALIST_BLOCK = {"emergency_c_section", "nicu", "icu", "trauma_emergency", "burn_care", "cardiac_surgery"}
SPECIALIST_NAME = re.compile(r"\beyes?\b|netra|nethra|ophthalm|retina|lasik|dental|dentist|\bteeth\b|\btooth\b|\bent\b|ear,? nose|\bskin\b|derma|\bhair\b|cosmetic|physiother|veterinar", re.I)
GENERAL_NAME = re.compile(r"matern|women|mother|\bchild|general|multi|nursing home|medical college|institute of medical", re.I)
IMAGING_LAB = re.compile(r"radiolog|imaging|patholog|anesthe|anaesthe|laborator")

SERVICES = {k: dict(s, source="curated", code=None, stems=None) for k, s in CURATED2.items()}
for code in counts.index:
    if not NOT_A_SERVICE.match(code):
        SERVICES["sp_" + code.replace(" ", "_")] = auto_service_v2(code)
for k, s in SERVICES.items():
    s["key"] = k
    s["text_re"] = re.compile(s["text"], re.I)
    s["equipment_re"] = re.compile(s["equipment"], re.I)
    s["specialty_re"] = re.compile(s["specialty"], re.I) if s["specialty"] else None
print(f"B/C) {len(SERVICES):,} services; example phrase patterns:",
      {k: SERVICES[k]["text"] for k in ["sp_pediatric_neurology", "sp_glaucoma_ophthalmology", "sp_cataract_and_anterior_segment_surgery"] if k in SERVICES})

# D, E, F) new checks
DUBIOUS = re.compile(r"energy quantum|quantum (healing|medicine|therap\w*|technolog\w*)|energy (medicine|healing)|\breiki\b|"
                     r"magnet(ic)? therapy|bio[- ]?resonance|miracle (cure|treatment|healing)|guaranteed cure|cures? (cancer|diabetes|all)", re.I)
AYUSH = re.compile(r"homo?eopath|ayurved|\bunani\b|\bsiddha\b|naturopath|panchakarm|chikits", re.I)
NAMED = re.compile(r"\b((?:[A-Z][A-Za-z'.&-]*\s+){1,4})(?:Hospitals?|Clinic|Nursing Home|Medical Cent(?:re|er)|"
                   r"Heart Cent(?:re|er)|Healthcare)\b")
GENERIC = set(("super speciality specialty specialities multi multispeciality multispecialty general city care medical "
               "health healthcare centre center institute research hospital hospitals clinic clinics nursing home "
               "government district civil private limited trust memorial children womens women maternity mother "
               "child eye heart cancer dental kidney new best advanced modern national international provides offers "
               "includes including features available equipped fully well state based teaching tertiary secondary "
               "primary bedded our this the and with for from at in college sciences science university").split())
def name_key(s):
    # the distinctive part of a name, spaces removed: "Belle Vue Clinic" -> "bellevue", "Vijay Hospital" -> "vijay"
    return "".join(w for w in re.findall(r"[a-z]+", nz(s)) if len(w) >= 3 and w not in GENERIC)
def other_hospital(lines, own_key):
    # flag only a NEAR-namesake (e.g. "Vijan Hospital" in the record of "Vijay Hospital"): scraped text from a similarly
    # named place. Clearly different names are usually referrals or partners, so they are not flagged.
    if len(own_key) < 4:
        return None
    for x in lines:
        for m in NAMED.finditer(x):
            k = name_key(m.group(1))
            if len(k) < 4 or k in own_key or own_key in k:
                continue
            if 0.6 <= difflib.SequenceMatcher(None, k, own_key).ratio() < 0.87:   # >= 0.87 is usually a spelling variant
                return m.group(0).strip()
    return None

SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")
QUANT = re.compile(r"\b\d+\s*-?\s*(bed(ded)?s?|machines?|units?|stations?|slices?|tesla|ventilators?|incubators?|warmers?|cots?|chairs?|theatres?|ots?)\b", re.I)
EMERG = re.compile(r"24\s*[x/*]\s*7|24 ?hours?|round[- ]the[- ]clock|emergency", re.I)
SERIOUS = ["flag_mixed_identity", "flag_state_mismatch", "flag_duplicate"]
PENALTIES = [("flag_mixed_identity", 30, "record may merge two different places"),
             ("flag_state_mismatch", 15, "address state conflicts with PIN code"),
             ("flag_duplicate", 10, "duplicate name in the same city"),
             ("flag_too_many_specialties", 10, "specialty list looks inflated"),
             ("flag_implausible_counts", 10, "doctor/bed numbers look unrealistic"),
             ("flag_coords_far_from_pin", 5, "coordinates disagree with PIN code")]

# 1) evidence per facility (same as cell 3, plus the AYUSH and name checks)
FAC = []
for r in silver.to_dict("records"):
    proc = as_list(r["procedure"]) + as_list(r["capability"])
    equip = as_list(r["equipment"])
    desc = [x for x in SENT_SPLIT.split(r["description"] or "") if x.strip()]
    codes = {norm_code(x) for x in as_list(r["specialties"])} - {""}
    ayush = bool(AYUSH.search(str(r["name"]))) or (len(codes) > 0 and sum(bool(AYUSH.search(c)) for c in codes) / len(codes) >= 0.5)
    FAC.append(dict(r=r, proc=proc, equip=equip, desc=desc, codes=codes, ayush=ayush, name_tok=name_key(r["name"]),
                    proc_n=[nz(x) for x in proc], equip_n=[nz(x) for x in equip], desc_n=[nz(x) for x in desc]))
print(f"E) {sum(f['ayush'] for f in FAC):,} facilities look like homeopathy/ayurveda/unani")

# 2) word index (same as cell 3)
postings = defaultdict(set)
for i, f in enumerate(FAC):
    for w in set(re.findall(r"[a-z]+", " ".join(f["proc_n"] + f["equip_n"] + f["desc_n"]))):
        postings[w].add(i)
vocab = sorted(postings)
code_to_fac = defaultdict(set)
for i, f in enumerate(FAC):
    for c in f["codes"]:
        code_to_fac[c].add(i)
def text_candidates(stems):
    out = None
    for s in stems:
        j, hit = bisect_left(vocab, s), set()
        while j < len(vocab) and vocab[j].startswith(s):
            hit |= postings[vocab[j]]
            j += 1
        out = hit if out is None else out & hit
        if not out:
            return set()
    return out or set()

# 3) scoring = v1 rules + fixes C-F
def score_pair_v2(f, s):
    r = f["r"]
    if s["source"] == "curated":
        hit_spec = sorted(c for c in f["codes"] if s["specialty_re"].search(c.replace(" ", ""))
                          and (s.get("key") in ("mri", "ct_scan") or not IMAGING_LAB.search(c)))
    else:
        hit_spec = [s["code"]] if s["code"] in f["codes"] else []
    tr, er = s["text_re"], s["equipment_re"]
    hit_proc = [o for o, n in zip(f["proc"], f["proc_n"]) if tr.search(n)]
    hit_equip = [o for o, n in zip(f["equip"], f["equip_n"]) if er.search(n) or tr.search(n)]
    hit_desc = [o for o, n in zip(f["desc"], f["desc_n"]) if tr.search(n)]
    ex = EXCLUDE.get(s.get("key"))
    if ex:
        hit_proc = [x for x in hit_proc if not ex.search(x)]
        hit_equip = [x for x in hit_equip if not ex.search(x)]
        hit_desc = [x for x in hit_desc if not ex.search(x)]
    dubious = [x for x in hit_proc + hit_equip + hit_desc if DUBIOUS.search(x)]
    if dubious:
        hit_proc = [x for x in hit_proc if not DUBIOUS.search(x)]
        hit_equip = [x for x in hit_equip if not DUBIOUS.search(x)]
        hit_desc = [x for x in hit_desc if not DUBIOUS.search(x)]
    if not (hit_spec or hit_proc or hit_equip or hit_desc or dubious):
        return None
    pts, why = 0, []
    if hit_proc: pts += 30; why.append(f"+30 procedure/capability mentions it ({len(hit_proc)})")
    if hit_equip: pts += 25; why.append(f"+25 equipment listed ({len(hit_equip)})")
    if hit_desc: pts += 15; why.append("+15 description mentions it")
    if hit_spec: pts += 10; why.append("+10 listed as a specialty")
    quantified = [x for x in hit_proc + hit_equip + hit_desc if QUANT.search(x)]
    if quantified: pts += 10; why.append("+10 concrete numbers in the evidence")
    multi = max(len(hit_proc), len(hit_equip), len(hit_desc)) >= 2
    if multi: pts += 10; why.append("+10 stated in several separate places")
    big = pd.notna(r["n_beds"]) and r["n_beds"] >= 200 and not r["flag_implausible_counts"]
    if big and (hit_proc or hit_equip or hit_desc) and (s["source"] == "curated" or s["hospital_level"]):
        pts += 10; why.append(f"+10 large hospital ({int(r['n_beds'])} beds)")
    if s["emergency"] and any(EMERG.search(x) for x in f["proc"] + f["desc"]):
        pts += 5; why.append("+5 24/7 or emergency wording")
    if pd.notna(r["n_beds"]) or pd.notna(r["n_doctors"]):
        pts += 5; why.append("+5 size known (beds/doctors)")
    pen, problems, ftype = 0, [], r["facility_type"]
    type_implausible = s["hospital_level"] and ftype in ("dentist", "pharmacy")
    if type_implausible: pen += 40; problems.append(f"-40 a {ftype} claiming a hospital-level service")
    specialist = (s.get("key") in SPECIALIST_BLOCK and not type_implausible
                  and bool(SPECIALIST_NAME.search(str(r["name"]))) and not GENERAL_NAME.search(str(r["name"])))
    if specialist: pen += 40; problems.append("-40 an eye/dental/ENT/skin specialist claiming an emergency or maternity service")
    if s["hospital_level"] and ftype == "clinic" and pd.isna(r["n_beds"]):
        pen += 15; problems.append("-15 clinic with no beds listed claims a hospital-level service")
    ayush = f["ayush"] and not AYUSH.search(s["label"])         # any modern service (incl. free-text searches)
    if ayush: pen += 30; problems.append("-30 a homeopathy/ayurveda/unani facility claiming a modern-medicine service")
    if dubious: pen += 20; problems.append(f"-20 pseudo-science wording in the claim: '{dubious[0].strip()[:80]}'")
    other = other_hospital(hit_proc + hit_desc + hit_equip, f["name_tok"])
    if other: pen += 20; problems.append(f"-20 evidence names a similarly-named but different facility ('{other}'), may describe another place")
    for flag, p, txt in PENALTIES:
        if r[flag]:
            pen += p; problems.append(f"-{p} {txt}")
    score = int(max(0, min(100, pts - pen)))
    n_hard = sum(bool(h) for h in (hit_proc, hit_equip, hit_desc))
    serious = type_implausible or specialist or ayush or bool(dubious) or any(r[x] for x in SERIOUS)
    kinds = n_hard + int(multi) + int(bool(quantified))      # H: independent kinds of support
    level = ("Verified" if score >= 60 and kinds >= 2 and not serious and not other
             else "Likely" if score >= 35 and n_hard >= 1 else "Claimed only")
    labelled = ([("procedure/capability", x) for x in hit_proc] + [("equipment", x) for x in hit_equip]
                + [("description", x) for x in hit_desc] + [("specialty", x) for x in hit_spec]
                + [("doubtful claim", x) for x in dubious])
    labelled.sort(key=lambda fx: (fx[0] == "doubtful claim", not QUANT.search(fx[1]), fx[0] == "specialty"))
    return dict(unique_id=r["unique_id"], trust_score=score, trust_level=level, n_hard_evidence=n_hard, evidence_kinds=int(kinds),
                ev_procedure=bool(hit_proc), ev_equipment=bool(hit_equip), ev_description=bool(hit_desc),
                ev_specialty=bool(hit_spec), ev_quantified=bool(quantified),
                flag_ayush_claim=bool(ayush), flag_dubious_claim=bool(dubious), flag_other_facility=bool(other),
                evidence=[f"{a}: {b.strip()[:240]}" for a, b in labelled[:5]],
                score_reasons=why, problems=problems, points_evidence=int(pts), points_penalty=int(pen))

# 4) run all services
t0, rows = datetime.datetime.now(), []
for skey, s in SERVICES.items():          # not "key": Step 4 uses key() for place names
    cand = range(len(FAC)) if s["source"] == "curated" else (code_to_fac[s["code"]] | text_candidates(s["stems"]))
    for i in cand:
        out = score_pair_v2(FAC[i], s)
        if out:
            out.update(service=skey, service_label=s["label"], service_source=s["source"])
            rows.append(out)
scores = pd.DataFrame(rows)
print(f"scored {len(scores):,} facility x service pairs in {(datetime.datetime.now() - t0).seconds}s")

# 5) attach display columns, check, save (same tables as v1, so the app picks up v2 automatically)
show_cols = ["unique_id", "name", "facility_type", "operator_type", "address", "city", "state", "pincode",
             "pin_district", "loc_lat", "loc_lon", "loc_source", "flag_location_uncertain",
             "official_phone", "official_website", "n_beds", "n_doctors", "n_flags"]
gold = scores.merge(silver[show_cols], on="unique_id", how="left")
assert not gold.duplicated(["unique_id", "service"]).any()
assert gold["evidence"].str.len().min() >= 1
# one plain-language sentence per row, for the app: level, why, the best quote, and every caution (the uncertainty)
def explain(x):
    head = {"Verified": f"Verified: {x.evidence_kinds} independent kinds of evidence in this facility's own records",
            "Likely": "Likely: direct evidence found, but not fully confirmed",
            "Claimed only": "Claimed only: listed, but no direct evidence found. Call to confirm before going"}[x.trust_level]
    quote = re.sub(r"^[a-z /]+:\s*", "", x.evidence[0])[:150] if len(x.evidence) else ""
    cautions = [re.sub(r"^-\d+\s*", "", p) for p in x.problems]
    if x.flag_location_uncertain:
        cautions.append("location is approximate")
    out = head + (f'. Evidence: "{quote}"' if quote else "") + "."
    return out + (" Caution: " + "; ".join(cautions) + "." if cautions else "")
gold["explanation"] = [explain(x) for x in gold.itertuples()]
gold["silver_source"] = "workspace.default.silver_facilities"
gold["gold_version"] = "v2"
gold["gold_built_at"] = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
GOLD = "workspace.default.gold_facility_services"
spark.createDataFrame(gold).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD)
cat = (gold.pivot_table(index=["service", "service_label", "service_source"], columns="trust_level",
                        values="unique_id", aggfunc="count", fill_value=0)
       .reindex(columns=["Verified", "Likely", "Claimed only"], fill_value=0).reset_index())
cat.columns = ["service", "service_label", "service_source", "n_verified", "n_likely", "n_claimed_only"]
cat["n_facilities"] = cat[["n_verified", "n_likely", "n_claimed_only"]].sum(axis=1)
cat = cat.sort_values(["service_source", "n_facilities"], ascending=[True, False])
spark.createDataFrame(cat).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.gold_services")

# table + column descriptions (shown in Catalog Explorer and to anyone who queries the table)
spark.sql(f"COMMENT ON TABLE {GOLD} IS 'Gold v2: one row per facility x service with a trust level (Verified / Likely / Claimed only), "
          "a 0-100 score, quoted evidence and cautions. Built by 05_gold_all_services Step 3 from silver_facilities. "
          "Checked by hand: on 12 unseen Verified rows 11 were correct and 0 wrong.'")
COLS = {
    "unique_id": "facility id (same as silver_facilities)",
    "service": "service key: curated name (e.g. dialysis) or sp_<specialty>",
    "service_label": "service name to show to users",
    "service_source": "curated = 10 hand-defined services; specialty = generated from a specialty name in the data",
    "trust_level": "Verified (score >= 60 and 2+ independent kinds of evidence, no serious problem) / Likely (score >= 35 and some direct evidence) / Claimed only",
    "trust_score": "0-100: evidence points minus penalties (see score_reasons and problems)",
    "n_hard_evidence": "how many of procedure / equipment / description mention the service (0-3)",
    "evidence_kinds": "independent kinds of support: n_hard_evidence + several separate statements + concrete numbers",
    "evidence": "up to 5 quotes from the facility record that support the claim, best first; each starts with the field it came from",
    "score_reasons": "the evidence points that were added",
    "problems": "the penalties / cautions (the uncertainty to show to users)",
    "explanation": "one plain-language sentence for users: level, best quote and all cautions",
    "flag_ayush_claim": "homeopathy / ayurveda / unani facility claiming a modern-medicine or hospital-level service",
    "flag_dubious_claim": "claim uses pseudo-science wording (e.g. energy quantum); that text is not counted as evidence",
    "flag_other_facility": "evidence names a similarly-named but different hospital (may describe another place)",
    "points_evidence": "sum of evidence points", "points_penalty": "sum of penalty points",
    "gold_version": "v2 = after three rounds of manual verification",
}
for col, txt in COLS.items():
    if col in gold.columns:
        spark.sql(f"ALTER TABLE {GOLD} ALTER COLUMN {col} COMMENT '{txt}'")
spark.sql("COMMENT ON TABLE workspace.default.gold_services IS 'One row per service: how many facilities are Verified / Likely / Claimed only. Built by 05_gold_all_services Step 3.'")
print("examples of the user sentence:")
for lvl in ["Verified", "Likely", "Claimed only"]:
    print(" ", gold.loc[gold["trust_level"] == lvl, "explanation"].iloc[0][:300])

# make Steps 4 and 5 (search) use v2
F, score_pair, auto_service = FAC, score_pair_v2, auto_service_v2

v1 = spark.table("workspace.default.gold_facility_services_v1").groupBy("trust_level").count().toPandas().set_index("trust_level")["count"]
v2 = gold["trust_level"].value_counts()
print(f"saved {GOLD} (v2): {len(gold):,} rows, {gold['service'].nunique():,} services, {gold['unique_id'].nunique():,} facilities")
print(pd.DataFrame({"v1": v1, "v2": v2}).reindex(["Verified", "Likely", "Claimed only"]).assign(change=lambda d: d.v2 - d.v1).to_string())
print("new flags:", {c: int(gold[c].sum()) for c in ["flag_ayush_claim", "flag_dubious_claim", "flag_other_facility"]})
print("examples of 'similarly-named but different facility':")
for x in gold.loc[gold["flag_other_facility"], ["name", "problems"]].drop_duplicates("name").head(8).itertuples():
    print("  ", x.name, "->", [p for p in x.problems if "different facility" in p][0][:140])

# Fix I check: rows that got the specialist penalty
_fi = gold[gold.problems.map(lambda ps: any("specialist claiming" in p for p in ps))]
print("Fix I (eye/dental/ENT/skin specialist claiming emergency or maternity care):", len(_fi), "rows |", _fi.trust_level.value_counts().to_dict())
print(_fi.groupby("service").size().to_dict())
print("levels now:", gold.trust_level.value_counts().to_dict())

A) removed, not a service: [('community medicine', 12), ('pharmacology', 11), ('physiology', 8), ('multispeciality', 5), ('forensic medicine and toxicology', 4), ('anatomy', 3), ('superspeciality', 2), ('independent divisions', 1), ('all specialities', 1)]
B/C) 2,531 services; example phrase patterns: {'sp_pediatric_neurology': '\\bpediat\\w*[ \\t\\-/&]+(?:[a-z]+[ \\t\\-/&]+){0,2}?neurol\\w*', 'sp_glaucoma_ophthalmology': '\\bglauc\\w*', 'sp_cataract_and_anterior_segment_surgery': '\\bcatar\\w*[ \\t\\-/&]+(?:[a-z]+[ \\t\\-/&]+){0,2}?anter\\w*[ \\t\\-/&]+(?:[a-z]+[ \\t\\-/&]+){0,2}?segme\\w*[ \\t\\-/&]+(?:[a-z]+[ \\t\\-/&]+){0,2}?surge\\w*'}
E) 325 facilities look like homeopathy/ayurveda/unani
scored 487,051 facility x service pairs in 114s
examples of the user sentence:
  Verified: 3 independent kinds of evidence in this facility's own records. Evidence: "Provides dialysis treatment at Nair Hospital, Mumbai Central".
  Likely: direct evidence found, but not fully confirmed. Evidence: 

In [0]:
# Step 4: search ANY service text + location
#   1) curated service whose keywords match the query (e.g. "kidney dialysis" -> Dialysis)
#   2) a specialty whose words match the query (e.g. "cataract surgery" -> Cataract And Anterior Segment Surgery)
#   3) otherwise a free-text search scored on the fly with the same rules (e.g. "kidney stone")
lookup = spark.table("workspace.default.gold_location_lookup").toPandas()
key = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
show = silver.set_index("unique_id")[show_cols[1:]]

def km(la1, lo1, la2, lo2):
    la1, lo1, la2, lo2 = (np.radians(x) for x in (la1, lo1, la2, lo2))
    a = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def locate(text):
    t = str(text).strip()
    m = re.fullmatch(r"\s*(-?\d+(\.\d+)?)\s*,\s*(-?\d+(\.\d+)?)\s*", t)
    if m:
        return float(m.group(1)), float(m.group(3)), "coordinates you entered"
    k = key(t)
    order = ["pin", "city", "district", "post_office"] if k.isdigit() else ["city", "district", "post_office", "pin"]
    for ptype in order:
        hit = lookup[(lookup["place_type"] == ptype) & (lookup["place_key"] == k)]
        if len(hit):
            h = hit.sort_values("n_points", ascending=False).iloc[0]
            return h["lat"], h["lon"], f"{ptype.replace('_', ' ')} '{h['place_name']}' ({h['state']})"
    return None

def find_service(query):
    q = nz(query).strip()
    for k, s in SERVICES.items():                                    # 1) curated
        if s["source"] == "curated" and (s["text_re"].search(q) or nz(s["label"]) in q):
            return k, f"matched the curated service '{s['label']}'"
    code = norm_code(query)
    q_stems = {stem(w) for w in code.split() if w not in STOP}
    best, best_n = None, 0
    for k, s in SERVICES.items():                                    # 2) specialty containing all query words (largest one)
        if s["source"] == "specialty" and q_stems and q_stems <= set(s["stems"]):
            n = len(code_to_fac[s["code"]])
            if n > best_n:
                best, best_n = k, n
    if best:
        return best, f"matched the specialty '{SERVICES[best]['label']}' (listed by {best_n} facilities)"
    return None, "no matching service name, so we searched the facility text for your words"

def adhoc_scores(query):
    s = auto_service(norm_code(query))
    s.update(label=query, source="free_text", code=None,
             text_re=re.compile(s["text"], re.I), equipment_re=re.compile(s["equipment"], re.I), specialty_re=None)
    out = []
    for i in text_candidates(s["stems"]):
        o = score_pair(F[i], s)
        if o:
            out.append(o)
    df = pd.DataFrame(out)
    return df.join(show, on="unique_id") if len(df) else df, s

def recommend(query, location, top_n=5, max_km=200):
    loc = locate(location)
    if loc is None:
        return f"Could not find '{location}'. Try a 6-digit PIN code or a nearby city.", None
    lat, lon, how = loc
    k, note = find_service(query)
    # always ALSO search the facility text for the user's words, then keep the best score per facility
    text_rows, s_text = adhoc_scores(query)
    parts = [gold[gold["service"] == k]] if k else []
    if len(text_rows):
        parts.append(text_rows)
        note += f"; plus a text search for '{query}' ({text_rows['unique_id'].nunique()} facilities mention it)"
    if not parts or not sum(len(p) for p in parts):
        return f"No facility mentions '{query}'. Try a broader term (e.g. a specialty).", None
    s = SERVICES[k] if k else s_text
    c = (pd.concat(parts, ignore_index=True).sort_values("trust_score", ascending=False)
           .drop_duplicates("unique_id").copy())
    c["distance_km"] = km(lat, lon, c["loc_lat"], c["loc_lon"]).round(1)
    c = c[c["distance_km"] <= max_km]
    c["rank_score"] = c["trust_score"] - (0.5 if s["emergency"] else 0.2) * c["distance_km"]
    strong = c[c["trust_level"] != "Claimed only"].sort_values("rank_score", ascending=False)
    picks = strong.head(top_n)
    if len(picks) < 3:
        picks = pd.concat([picks, c[c["trust_level"] == "Claimed only"].sort_values("distance_km").head(3 - len(picks))])
    return f"'{query}' near {location}: {note}; searched around the {how}; {len(strong)} trusted options within {max_km} km.", picks

for q, place in [("dialysis", "Patna"), ("cataract surgery", "Chennai"), ("knee replacement", "Pune"), ("kidney stone", "Jaipur")]:
    header, picks = recommend(q, place)
    print("=" * 100 + "\n" + header)
    if picks is None:
        continue
    for i, (_, r) in enumerate(picks.iterrows(), 1):
        approx = " (location approximate)" if r["flag_location_uncertain"] else ""
        print(f"  {i}. {r['name']} | {r['trust_level']} {r['trust_score']}/100 | {r['distance_km']} km{approx}")
        print(f"     evidence: {r['evidence'][0][:150]}")

'dialysis' near Patna: matched the curated service 'Dialysis'; plus a text search for 'dialysis' (1002 facilities mention it); searched around the city 'Patna' (Bihar); 39 trusted options within 200 km.
  1. Hi-Tech Emergency Hospital – Saguna More, Bailey Road, Patna-801503 | Verified 100/100 | 10.3 km
     evidence: description: It has ICU (35 beds), ICCU (15 beds), NICU (15 beds), CT scan, digital X-ray, ultrasound with color Doppler, Echo, dialysis, mammography,
  2. Nectar Hospital | Verified 90/100 | 0.8 km
     evidence: procedure/capability: Performs dialysis treatment
  3. Patna Dialysis Centre | Verified 90/100 | 1.0 km
     evidence: procedure/capability: Hemodialysis is performed three or more times a week
  4. City Dialysis Centre | Verified 90/100 | 5.1 km
     evidence: procedure/capability: Provides hemodialysis services
  5. Indira Gandhi Institute of Medical Sciences | Verified 90/100 | 5.3 km
     evidence: procedure/capability: Offers Nephrology and Urology includin

In [0]:
# Step 5: SEARCH BOX for review. Type a service and a place in the boxes at the top of the notebook, then run this cell.
dbutils.widgets.text("service", "dialysis", "1. Service (any words)")
dbutils.widgets.text("location", "Patna", "2. Place (city, district or 6-digit PIN)")
dbutils.widgets.dropdown("top_n", "10", ["5", "10", "20"], "3. How many results")

q, place, n = dbutils.widgets.get("service"), dbutils.widgets.get("location"), int(dbutils.widgets.get("top_n"))
header, picks = recommend(q, place, top_n=n)
print(header)
if picks is not None:
    view = picks.assign(
        rank=range(1, len(picks) + 1),
        location_note=np.where(picks["flag_location_uncertain"], "approximate", ""),
        evidence_1=picks["evidence"].str[0],
        evidence_2=picks["evidence"].apply(lambda e: e[1] if len(e) > 1 else ""),
        why_score=picks["score_reasons"].str.join("; "),
        cautions=picks["problems"].str.join("; "),
    )[["rank", "name", "trust_level", "trust_score", "distance_km", "location_note", "city", "state",
       "official_phone", "evidence_1", "evidence_2", "why_score", "cautions", "facility_type"]]
    display(view)

'dialysis' near Patna: matched the curated service 'Dialysis'; plus a text search for 'dialysis' (1002 facilities mention it); searched around the city 'Patna' (Bihar); 39 trusted options within 200 km.


rank,name,trust_level,trust_score,distance_km,location_note,city,state,official_phone,evidence_1,evidence_2,why_score,cautions,facility_type
1,"Hi-Tech Emergency Hospital – Saguna More, Bailey Road, Patna-801503",Verified,100,10.3,,Patna,Bihar,+918102262107,"description: It has ICU (35 beds), ICCU (15 beds), NICU (15 beds), CT scan, digital X-ray, ultrasound with color Doppler, Echo, dialysis, mammography, radiology, pathology, physiotherapy and a Blood Centre.",procedure/capability: Advanced Kidney Dialysis,+30 procedure/capability mentions it (1); +25 equipment listed (1); +15 description mentions it; +10 listed as a specialty; +10 concrete numbers in the evidence; +10 large hospital (200 beds); +5 size known (beds/doctors),,hospital
2,Nectar Hospital,Verified,90,0.8,,Patna,Bihar,+919155990993,procedure/capability: Performs dialysis treatment,procedure/capability: Provides dialysis treatment in a dedicated dialysis unit,+30 procedure/capability mentions it (2); +25 equipment listed (1); +10 listed as a specialty; +10 stated in several separate places; +10 large hospital (2200 beds); +5 size known (beds/doctors),,hospital
3,Patna Dialysis Centre,Verified,90,1.0,,Patna,Bihar,+918541877685,procedure/capability: Hemodialysis is performed three or more times a week,procedure/capability: Dialysis treatment is conducted by a dialysis technician under the guidance of a nephrologist,+30 procedure/capability mentions it (14); +25 equipment listed (5); +15 description mentions it; +10 listed as a specialty; +10 stated in several separate places,,clinic
4,City Dialysis Centre,Verified,90,5.1,,Patna,Bihar,+919771465188,procedure/capability: Provides hemodialysis services,procedure/capability: Offers CAPD (Peritoneal Dialysis),+30 procedure/capability mentions it (6); +25 equipment listed (2); +15 description mentions it; +10 listed as a specialty; +10 stated in several separate places,,clinic
5,Indira Gandhi Institute of Medical Sciences,Verified,90,5.3,,Patna,Bihar,+916122297631,procedure/capability: Offers Nephrology and Urology including renal dialysis and renal transplantation,procedure/capability: Dialysis is provided at the IGIMS Dialysis Unit,+30 procedure/capability mentions it (2); +25 equipment listed (2); +10 listed as a specialty; +10 stated in several separate places; +10 large hospital (1000 beds); +5 size known (beds/doctors),,hospital
6,Netaji Subhas Medical College & Hospital,Verified,90,29.4,,Bihta,Bihar,+918448443305,procedure/capability: Hemodialysis (HD) and haemodiafiltration (HDF),procedure/capability: Continuous renal replacement therapy (CRRT) and prolonged intermittent renal replacement therapy (PIRRT),+30 procedure/capability mentions it (8); +25 equipment listed (1); +10 listed as a specialty; +10 stated in several separate places; +10 large hospital (625 beds); +5 size known (beds/doctors),,hospital
7,Rajeshwar Hospital,Verified,80,2.7,,Patna,Bihar,+918877700023,procedure/capability: Dialysis and Continuous Renal Replacement Therapy (CRRT),procedure/capability: Renal Replacement Therapy facility (CRRT),+30 procedure/capability mentions it (2); +25 equipment listed (1); +10 listed as a specialty; +10 stated in several separate places; +5 size known (beds/doctors),,hospital
8,Big Apollo Spectra Hospitals,Verified,80,2.9,,Patna,Bihar,+918800001068,procedure/capability: Provides hemodialysis treatment in a state-of-the-art dialysis wing with ultra-pure water,equipment: State-of-the-art dialysis wing with hemodialysis equipment and ultra-pure water,+30 procedure/capability mentions it (1); +25 equipment listed (6); +10 listed as a specialty; +10 stated in several separate places; +5 size known (beds/doctors),,hospital
9,Samarpan Trauma and Research Centre,Verified,80,3.4,,Patna,Bihar,+919304111833,procedure/capability: Dialysis/haemodialysis,procedure/capability: Dialysis/hemodialysis,+30 procedure/capability mentions it (3); +25 equipment listed (1); +10 listed as a specialty; +10 stated in several separate place

In [0]:
# Step 6: VERIFICATION round 1 = stratified sample of 100 facility x service rows (drawn from Gold v1) for manual review
# Safe to re-run: once verify_sample_v1 exists it is loaded, never re-drawn (the human verdicts refer to these exact rows).
from pyspark.sql import functions as SF, Window      # SF, not F: F is the facility list used by the search steps
if spark.catalog.tableExists("workspace.default.verify_sample_v1"):
    out = spark.table("workspace.default.verify_sample_v1").toPandas().sort_values("sample_id")
    print("loaded existing workspace.default.verify_sample_v1:", len(out), "rows")
else:

    g = spark.table("workspace.default.gold_facility_services")
    s = spark.table("workspace.default.silver_facilities").select("unique_id", "websites", "source_urls")
    PLAN = {"Verified": 40, "Likely": 35, "Claimed only": 25}      # weighted toward what users actually see
    parts = []
    for level, n in PLAN.items():
        for src in ["curated", "specialty"]:
            k = n // 2 if src == "curated" else n - n // 2
            per_service = 3 if src == "curated" else 1                    # spread across many different services
            w = Window.partitionBy("service").orderBy(SF.rand(42))
            parts.append(g.filter((SF.col("trust_level") == level) & (SF.col("service_source") == src))
                          .withColumn("rn", SF.row_number().over(w)).filter(SF.col("rn") <= per_service)
                          .orderBy(SF.rand(7)).limit(k))
    sample = parts[0]
    for p in parts[1:]:
        sample = sample.unionByName(p)

    pdf = sample.join(s, "unique_id", "left").toPandas().sample(frac=1, random_state=3).reset_index(drop=True)
    pdf.insert(0, "sample_id", range(1, len(pdf) + 1))
    OVERLAP = set(range(46, 51)) | set(range(96, 101))
    pdf.insert(1, "reviewer", ["Both" if i in OVERLAP else ("Joyce" if i <= 50 else "Tim") for i in pdf["sample_id"]])
    ev = pdf["evidence"].apply(list)
    for j in range(3):
        pdf[f"evidence_{j + 1}"] = ev.apply(lambda e: e[j] if len(e) > j else "")
    pdf["how_scored"] = pdf["score_reasons"].apply(lambda x: "; ".join(x))
    pdf["cautions"] = pdf["problems"].apply(lambda x: "; ".join(x))
    pdf["links_to_check"] = pdf.apply(lambda r: " ".join(([r["official_website"]] if r["official_website"] else [])
                                                         + list(r["source_urls"] if r["source_urls"] is not None else [])[:3]), axis=1)
    for c in ["verdict_joyce", "reason_joyce", "verdict_tim", "reason_tim"]:
        pdf[c] = ""

    out = pdf[["sample_id", "reviewer", "service_label", "service_source", "name", "facility_type", "city", "state",
               "trust_level", "trust_score", "evidence_1", "evidence_2", "evidence_3", "how_scored", "cautions",
               "links_to_check", "verdict_joyce", "reason_joyce", "verdict_tim", "reason_tim", "unique_id", "service"]]
    spark.createDataFrame(out).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.verify_sample_v1")

    print(f"saved workspace.default.verify_sample_v1: {len(out)} rows, {out['service'].nunique()} different services")
    print("by trust level:", out["trust_level"].value_counts().to_dict())
    print("by reviewer:", out["reviewer"].value_counts().to_dict(), "(Both = reviewed by Joyce AND Tim)")
display(out)

loaded existing workspace.default.verify_sample_v1: 100 rows


sample_id,reviewer,service_label,service_source,name,facility_type,city,state,trust_level,trust_score,evidence_1,evidence_2,evidence_3,how_scored,cautions,links_to_check,verdict_joyce,reason_joyce,verdict_tim,reason_tim,unique_id,service
1,Joyce,Physiology,specialty,C U Shah Medical College,hospital,Surendranagar,Gujarat,Claimed only,30,"procedure/capability: Basic Sciences departments include Anatomy, Physiology, Biochemistry",,,+30 procedure/capability mentions it (1),,cusmc.org https://cusmc.org/faculty/39/SURGERY https://cusmc.org/contact-us https://cusmc.org/,,,,,05191883-9634-46be-a1e9-0dad02b33f69,sp_physiology
2,Joyce,Keratoplasty,specialty,Paras HMRI Hospital,hospital,Patna,Bihar,Likely,35,procedure/capability: Cornea Transplant (Keratoplasty),,,+30 procedure/capability mentions it (1); +5 size known (beds/doctors),,parashospitals.com https://www.makemytrip.com/hotels/hotels-in-west_ram_krishna_nagar_patna-patna.html https://pmc.ncbi.nlm.nih.gov/articles/PMC7980235/ https://www.hexahealth.com/patna/hospital/shivam-hospital-research-institute-patna,,,,,eb938888-78b6-4b09-9560-4847f46bd1e7,sp_keratoplasty
3,Joyce,Newborn intensive care (NICU),curated,Satish Nursing Home,hospital,Mainpuri,Uttar Pradesh,Verified,70,"description: A 5-storey facility with 50 beds, inaugurated in 2020, offers 24/7 emergency services, ICU, operation theater, delivery room, NICU, burn ward, general and private wards, oxygen plant, and an upcoming dialysis wing.",procedure/capability: Provides neonatal care in Neonatal Intensive Care Unit (NICU),procedure/capability: Neonatal Intensive Care Unit (NICU),+30 procedure/capability mentions it (2); +15 description mentions it; +10 listed as a specialty; +10 concrete numbers in the evidence; +5 24/7 or emergency wording; +5 size known (beds/doctors),-5 coordinates disagree with PIN code,satishhospital.com https://www.satishhospital.com/about-us/ http://www.satishhospital.com/ https://www.facebook.com/108144027760452,,,,,72b1bbcd-4cc4-403f-9df8-6a44ac724885,nicu
4,Joyce,Pediatric Cardiologist,specialty,Sree Sudheendra Medical Mission Hospital,hospital,Ernakulam,Kerala,Likely,35,"procedure/capability: Departments include General Medicine, Endocrinology, Gynecology, Paediatrics, Orthopaedics, Rheumatology, Cardiology, ENT, Anaesthesiology, Casualty, Dental, Dermatology, Gastroenterology, Surgical Gastroenterology, Gastro & Hepato Biliary",,,+30 procedure/capability mentions it (1); +5 size known (beds/doctors),,ssmmhospital.com https://www.justdial.com/Ernakulam/Hospitals/nct-10253670 https://ssmmhospital.com/Overview https://karetrip.com/hospital/sree-sudheendra-medical-mission-hospital,,,,,eed8db66-bbf8-4669-be2b-d4e4880f2522,sp_pediatric_cardiologist
5,Joyce,Cardiothoracic And Cardiovascular Surgery,specialty,Kims Hospital,hospital,Hyderabad,Telangana,Claimed only,25,procedure/capability: Cardiothoracic and Cardiovascular Surgery services are available at KIMS Hospital Hyderabad,,,+30 procedure/capability mentions it (1); +5 size known (beds/doctors),-10 doctor/bed numbers look unrealistic,kimshospitals.com https://moovitapp.com/index/en/public_transit-Medplus_Madinaguda_My_Home_Jewel-Hyderabad-site_206008496-5997 https://pmc.ncbi.nlm.nih.gov/articles/PMC8556848/ https://www.godigit.com/guides/hospitals/tb-hospitals-in-hyderabad,,,,,60d478ce-f2fd-4981-b18a-292785fdc401,sp_cardiothoracic_and_cardiovascular_surgery
6,Joyce,Burn care,curated,Dwivedi Hospital,hospital,Kanpur,Uttar Pradesh,Claimed only,15,specialty: burn and trauma plastic surgery,,,+10 listed as a specialty; +5 24/7 or emergency wording; +5 size known (beds/doctors),-5 coordinates disagree with PIN code,https://www.medicineindia.org/hospitals-in-city/kanpur-uttar-pradesh https://www.proptiger.com/kanpur/kalyanpur-53134/hospitals https://www.doctoriduniya.com/hospital/kanpur/emerald-hospital-kanpur,,,,,665ed5c5-ce1e-48c7-8e8f-121413a87845,burn_care
7,Joyce,Accident And Trauma,specialty,Sai Samarth Hospital,hospital,Navi Mumbai,Maharashtra

In [0]:
# Step 7: BEFORE (v1) vs AFTER (v2) on the same 100 hand-checked rows
# Final human verdict per sample_id 1..100, copied from the "DSC Gold Verification" Google Sheet (reasons stay in the sheet).
# Rule: Joyce's rows -> Joyce, Tim's rows -> Tim, Both rows -> Joyce (Joyce and Tim agreed on 9 of the 10 Both rows).
# Y = Yes, N = No, U = Unsure, A = N/A (not a service), - = not reviewed
V = "AYYUYUYYYYUYYNYUYYNNYYUYUUUUYUYYUYNYUYNUYYYYYUNYYYUYUYUU-UYUYYNUUYYUYYYYUNYYYYYYUYUYYYYYUYUUUUYNUYYY"
assert len(V) == 100
CODE = {"Y": "Yes", "N": "No", "U": "Unsure", "A": "N/A", "-": None}
vs = (spark.table("workspace.default.verify_sample_v1")
      .select("sample_id", "reviewer", "unique_id", "service", "service_label", "service_source", "trust_level", "trust_score")
      .toPandas().sort_values("sample_id"))
vs["verdict"] = vs["sample_id"].map(lambda i: CODE[V[i - 1]])
spark.createDataFrame(vs).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.verify_results_v1")
print("saved workspace.default.verify_results_v1 (human verdicts):", vs["verdict"].value_counts(dropna=False).to_dict())

g2 = (spark.table("workspace.default.gold_facility_services")
      .select("unique_id", "service", "trust_level", "trust_score", "problems").toPandas())
m = (vs.rename(columns={"trust_level": "level_v1", "trust_score": "score_v1"})
       .merge(g2.rename(columns={"trust_level": "level_v2", "trust_score": "score_v2"}), on=["unique_id", "service"], how="left"))
m["level_v2"] = m["level_v2"].fillna("Removed")
judged = m[m["verdict"].isin(["Yes", "No", "Unsure"])]

def precision_table(col, levels):
    t = (pd.crosstab(judged[col], judged["verdict"])
         .reindex(index=levels, columns=["Yes", "No", "Unsure"], fill_value=0))
    t["rows"] = t[["Yes", "No", "Unsure"]].sum(axis=1)
    t["% Yes"] = (100 * t["Yes"] / t["rows"]).round(0)
    t["% No"] = (100 * t["No"] / t["rows"]).round(0)
    return t
print("\nBEFORE (v1): what the human reviewers found, by the level the system gave")
print(precision_table("level_v1", ["Verified", "Likely", "Claimed only"]).to_string())
print("\nAFTER (v2): same rows, re-scored with the fixes")
print(precision_table("level_v2", ["Verified", "Likely", "Claimed only", "Removed"]).to_string())

wrong = judged[judged["verdict"] == "No"]
print(f"\nThe {len(wrong)} rows humans marked WRONG: v1 level -> v2 level")
for x in wrong.itertuples():
    print(f"  #{x.sample_id:>3} {x.service_label[:35]:35} {x.level_v1:13} -> {x.level_v2:13} {list(x.problems)[:2] if x.problems is not None and not isinstance(x.problems, float) else ''}")
lost = judged[(judged["verdict"] == "Yes") & (judged["level_v1"] != judged["level_v2"])]
print(f"\nSide effects: {len(lost)} rows humans marked CORRECT changed level:")
for x in lost.itertuples():
    print(f"  #{x.sample_id:>3} {x.service_label[:35]:35} {x.level_v1:13} -> {x.level_v2:13} {list(x.problems)[:2] if x.problems is not None and not isinstance(x.problems, float) else ''}")
na = m[m["verdict"] == "N/A"]
print("\nNot-a-service rows:", [(x.service_label, x.level_v1, "->", x.level_v2) for x in na.itertuples()])


saved workspace.default.verify_results_v1 (human verdicts): {'Yes': 57, 'Unsure': 32, 'No': 9, 'N/A': 1, None: 1}

BEFORE (v1): what the human reviewers found, by the level the system gave
verdict       Yes  No  Unsure  rows  % Yes  % No
level_v1                                        
Verified       30   3       7    40   75.0   8.0
Likely         19   2      13    34   56.0   6.0
Claimed only    8   4      12    24   33.0  17.0

AFTER (v2): same rows, re-scored with the fixes
verdict       Yes  No  Unsure  rows  % Yes  % No
level_v2                                        
Verified       31   2       7    40   78.0   5.0
Likely         18   0      10    28   64.0   0.0
Claimed only    8   4      12    24   33.0  17.0
Removed         0   3       3     6    0.0  50.0

The 9 rows humans marked WRONG: v1 level -> v2 level
  # 14 Immuno Haematology                  Claimed only  -> Removed       
  # 19 Pediatric Neurology                 Likely        -> Removed       
  # 20 Cardiac surg

In [0]:
# Step 8: VERIFICATION round 2 = 30 NEW rows from Gold v2 (none of the first 100), to get a FAIR accuracy number for v2
# (the v2 fixes were designed by looking at the first 100 rows, so those rows over-state how good v2 is)
# Safe to re-run: once verify_sample_v2 exists it is loaded, never re-drawn.
from pyspark.sql import functions as SF, Window      # SF, not F: F is the facility list used by the search cells
T2 = "workspace.default.verify_sample_v2"
if spark.catalog.tableExists(T2):
    out2 = spark.table(T2).toPandas().sort_values("sample_id")
    print("loaded existing", T2)
else:
    g = spark.table("workspace.default.gold_facility_services").filter("gold_version = 'v2'")
    g = g.join(spark.table("workspace.default.verify_sample_v1").select("unique_id", "service"), ["unique_id", "service"], "left_anti")
    src = spark.table("workspace.default.silver_facilities").select("unique_id", "source_urls")
    PLAN = {"Verified": 12, "Likely": 10, "Claimed only": 8}       # same mix as round 1, scaled to 30
    parts = []
    for level, n in PLAN.items():
        for kind in ["curated", "specialty"]:
            k = n // 2 if kind == "curated" else n - n // 2
            w = Window.partitionBy("service").orderBy(SF.rand(2026))
            parts.append(g.filter((SF.col("trust_level") == level) & (SF.col("service_source") == kind))
                          .withColumn("rn", SF.row_number().over(w)).filter(SF.col("rn") <= 1)   # 1 per service: many services
                          .orderBy(SF.rand(11)).limit(k))
    smp = parts[0]
    for p in parts[1:]:
        smp = smp.unionByName(p)
    pdf = smp.join(src, "unique_id", "left").toPandas().sample(frac=1, random_state=5).reset_index(drop=True)
    pdf.insert(0, "sample_id", range(101, 101 + len(pdf)))
    pdf.insert(1, "reviewer", ["Joyce" if i < 116 else "Tim" for i in pdf["sample_id"]])
    ev = pdf["evidence"].apply(list)
    for j in range(3):
        pdf[f"evidence_{j + 1}"] = ev.apply(lambda e: e[j] if len(e) > j else "")
    pdf["how_scored"] = pdf["score_reasons"].apply(lambda x: "; ".join(x))
    pdf["cautions"] = pdf["problems"].apply(lambda x: "; ".join(x))
    pdf["links_to_check"] = pdf.apply(lambda r: " ".join(([r["official_website"]] if r["official_website"] else [])
                                                         + list(r["source_urls"] if r["source_urls"] is not None else [])[:3]), axis=1)
    for c in ["verdict_joyce", "reason_joyce", "web_joyce", "verdict_tim", "reason_tim", "web_tim"]:
        pdf[c] = ""
    out2 = pdf[["sample_id", "reviewer", "service_label", "service_source", "name", "facility_type", "city", "state",
                "evidence_1", "evidence_2", "evidence_3", "links_to_check",
                "verdict_joyce", "reason_joyce", "web_joyce", "verdict_tim", "reason_tim", "web_tim",
                "trust_level", "trust_score", "how_scored", "cautions", "unique_id", "service"]]
    spark.createDataFrame(out2).write.saveAsTable(T2)
    print("saved", T2)
print(f"{len(out2)} rows, {out2['service'].nunique()} different services")
print("by trust level:", out2["trust_level"].value_counts().to_dict(), "| by reviewer:", out2["reviewer"].value_counts().to_dict())
print("trust_level, trust_score, how_scored and cautions are the LAST columns: hide them while reviewing")
display(out2)


loaded existing workspace.default.verify_sample_v2
30 rows, 21 different services
by trust level: {'Verified': 12, 'Likely': 10, 'Claimed only': 8} | by reviewer: {'Joyce': 15, 'Tim': 15}
trust_level, trust_score, how_scored and cautions are the LAST columns: hide them while reviewing


sample_id,reviewer,service_label,service_source,name,facility_type,city,state,evidence_1,evidence_2,evidence_3,links_to_check,verdict_joyce,reason_joyce,web_joyce,verdict_tim,reason_tim,web_tim,trust_level,trust_score,how_scored,cautions,unique_id,service
101,Joyce,Homeopath,specialty,The Apple Clinic - Super Specialty Homoeopathy,clinic,Chandigarh,Chandigarh,procedure/capability: Offers both homeopathic and conventional healthcare services,procedure/capability: Specializes in homeopathy,procedure/capability: Provides both homeopathic and conventional medical services,theappleclinic.com https://www.drlogy.com/chandigarh/clinic/the-apple-clinic---super-specialty-homoeopathy https://theappleclinic.com/ http://www.theappleclinic.com/,,,,,,,Verified,75,+30 procedure/capability mentions it (10); +25 equipment listed (2); +15 description mentions it; +5 size known (beds/doctors),,6fb1d108-f68d-4d42-acce-e6b911803e07,sp_homeopath
102,Joyce,Organ Transplantation,specialty,Amrita Hospital,hospital,Kochi,Kerala,"description: Gastrointestinal surgery & solid organ transplantation at Amrita Hospital, Kochi is a premier medical facility dedicated to advanced care for complex gastrointestinal, liver, gallbladder, and pancreatic diseases.",,,amritahospital.org https://www.amritahospitals.org/kochi/doctor/mohan-abraham https://www.amritahospitals.org/kochi/department/centres-of-excellences/digestive-diseases/gastrointestinal-surgery https://www.justdial.com/Pathanamthitta/Hospitals-in-Lahai/nct-10253670,,,,,,,Claimed only,20,+15 description mentions it; +5 size known (beds/doctors),,33af2594-ec2f-4268-a1a6-9cb47b04d798,sp_organ_transplantation
103,Joyce,Cancer treatment (chemo / radiation),curated,YASHONANDAN HOSPITAL,hospital,Thane,Maharashtra,specialty: medical oncology,,,yashonandanhospital.com https://www.practo.com/thane/clinic/dento-sculpt-multi-speciality-dental-clinic-panch-pakhadi/doctors https://www.practo.com/thane/clinic/uttam-microdentistry-implant-centre-mumbai https://www.clinicspots.com/hospital/yashonandan-hospital,,,,,,,Claimed only,5,+10 listed as a specialty; +5 size known (beds/doctors),-10 specialty list looks inflated,1c776e87-e373-444d-a2db-ca79499457d2,cancer_treatment
104,Joyce,Trauma / emergency care,curated,Daya Hospital,hospital,Faridabad,Haryana,specialty: emergency medicine,,,https://www.sarvodayahospital.com/ https://www.justdial.com/Faridabad/Daya-Hospital-Khera-Sarai/011PXX11-XX11-250130191346-C9A4_BZDET https://www.justdial.com/Faridabad/Dayal-Hospital-Faridabad-Sector-53/011PXX11-XX11-170711235855-T2T9_BZDET https://www.justdial.com/Sirsa-Haryana/Dayal-Hospital-Near-Sangwan-Circle/9999P1666-1666-241202194344-D8J6_BZDET,,,,,,,Claimed only,10,+10 listed as a specialty; +5 24/7 or emergency wording,-5 coordinates disagree with PIN code,1797aa65-f92d-486a-a53c-11ea65a8aafe,trauma_emergency
105,Joyce,Smile Design,specialty,Orthocare Dental Speciality Centre,dentist,Dehradun,Uttarakhand,"procedure/capability: Smile designing using braces, porcelain veneers, and zirconia crowns","description: Situated in the heart of Dehradun, Orthocare Dental Speciality Centre is a premier dental care clinic/centre offering services including braces (traditional, invisible), Invisalign, implants, root canal, fillings, scaling, gum treatment, wh",,orthocare.net.in http://www.orthocare.net.in/ https://dir.indiamart.com/dehradun/dental-implant-surgery.html https://www.facebook.com/395001423925116,,,,,,,Likely,50,+30 procedure/capability mentions it (1); +15 description mentions it; +5 size known (beds/doctors),,eabf58d7-3d10-43cd-ad1b-4e3f6289d89e,sp_smile_design
106,Joyce,Endocrinology,specialty,AMRI Hospital,hospital,Kolkata,West Bengal,"procedure/capability: Specialises in Cardiology, Endocrinology, Urology, Oncology, Neurology, Hepatology, ENT surgery, Pulmonology, Nephrology, Bariatric surgery, and Orthopedics","procedure/capability: Endocrinology services available at AMRI Hospitals Salt Lake, Kolkata","procedure/capability: A

In [0]:
# Step 9: ROUND 2 result = v2 accuracy on 30 rows nobody had seen while building v2 (the fair number)
# Verdicts copied from the round-2 Google Sheet (sample_id 101..130; Joyce 101-115, Tim 116-130). Reasons stay in the sheet.
import pandas as pd
V2 = "YYUNYYYNYUUYUYYUNYUUYYYYYUYNYU"          # Y yes, N no, U unsure
WEB = "CCFFCCCFCSFCSCCFFCFSCCCCCFCFCF"         # C website confirms, F website found but silent, S no working website
CODE = {"Y": "Yes", "N": "No", "U": "Unsure"}
WCODE = {"C": "confirmed", "F": "not-found", "S": "no-site"}
r2 = spark.table("workspace.default.verify_sample_v2").select("sample_id", "reviewer", "unique_id", "service", "service_label",
                                                              "trust_level", "trust_score").toPandas().sort_values("sample_id")
assert len(V2) == len(WEB) == len(r2) == 30
r2["verdict"] = [CODE[c] for c in V2]
r2["web"] = [WCODE[c] for c in WEB]
spark.createDataFrame(r2).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.verify_results_v2")
t = pd.crosstab(r2["trust_level"], r2["verdict"]).reindex(index=["Verified", "Likely", "Claimed only"], columns=["Yes", "No", "Unsure"], fill_value=0)
t["rows"] = t.sum(axis=1)
t["% Yes"] = (100 * t["Yes"] / t["rows"]).round(0)
t["% No"] = (100 * t["No"] / t["rows"]).round(0)
print("ROUND 2 (unseen rows, Gold v2):")
print(t.to_string())
print("\nwebsite check:", r2["web"].value_counts().to_dict())


ROUND 2 (unseen rows, Gold v2):
verdict       Yes  No  Unsure  rows  % Yes  % No
trust_level                                     
Verified       11   0       1    12   92.0   0.0
Likely          4   3       3    10   40.0  30.0
Claimed only    2   1       5     8   25.0  12.0

website check: {'confirmed': 17, 'not-found': 10, 'no-site': 3}


In [0]:
# Step 10: CHECK after the round-2 fixes: all 130 hand-checked rows against the CURRENT gold table
# (round 2 is no longer an unseen test once we fix what it found; its fair number is the one in Step 9)
import pandas as pd
cur = spark.table("workspace.default.gold_facility_services").select("unique_id", "service", "trust_level").toPandas()
for name, tbl in [("ROUND 1 (rows 1-100, level at the time = v1)", "verify_results_v1"),
                  ("ROUND 2 (rows 101-130, level at the time = v2 before these fixes)", "verify_results_v2")]:
    r = spark.table(f"workspace.default.{tbl}").toPandas()
    r = r[r["verdict"].isin(["Yes", "No", "Unsure"])]
    r = r.merge(cur.rename(columns={"trust_level": "level_now"}), on=["unique_id", "service"], how="left")
    r["level_now"] = r["level_now"].fillna("Removed")
    t = (pd.crosstab(r["level_now"], r["verdict"])
         .reindex(index=["Verified", "Likely", "Claimed only", "Removed"], columns=["Yes", "No", "Unsure"], fill_value=0))
    t["rows"] = t.sum(axis=1)
    t["% Yes"] = (100 * t["Yes"] / t["rows"]).round(0)
    t["% No"] = (100 * t["No"] / t["rows"]).round(0)
    print(f"\n{name}: by the level the system gives NOW")
    print(t.to_string())
    ch = r[r["trust_level"] != r["level_now"]].sort_values("sample_id")
    print("rows whose level changed (human verdict: then -> now):")
    for x in ch.itertuples():
        print(f"  #{x.sample_id:>3} {x.verdict:6} {x.service_label[:38]:38} {x.trust_level:13} -> {x.level_now}")



ROUND 1 (rows 1-100, level at the time = v1): by the level the system gives NOW
verdict       Yes  No  Unsure  rows  % Yes  % No
level_now                                       
Verified       31   2       7    40   78.0   5.0
Likely         18   0      10    28   64.0   0.0
Claimed only    8   4      12    24   33.0  17.0
Removed         0   3       3     6    0.0  50.0
rows whose level changed (human verdict: then -> now):
  #  4 Unsure Pediatric Cardiologist                 Likely        -> Removed
  # 14 No     Immuno Haematology                     Claimed only  -> Removed
  # 16 Unsure Cancer treatment (chemo / radiation)   Claimed only  -> Removed
  # 19 No     Pediatric Neurology                    Likely        -> Removed
  # 20 No     Cardiac surgery                        Likely        -> Claimed only
  # 27 Unsure Cancer treatment (chemo / radiation)   Verified      -> Claimed only
  # 30 Unsure Intensive care (ICU)                   Likely        -> Claimed only
  # 40 Un

In [0]:
# Step 11: FAMOUS-HOSPITAL TEST. These hospitals CERTAINLY offer these services (public knowledge).
# Does the system find them, what level does it give, and where do they rank when you search that service in their city?
# Needs Steps 1-4 (scoring + search) to have run in this session.
import pandas as pd
QUERY = {"cardiac_surgery": "cardiac surgery", "icu": "icu", "trauma_emergency": "emergency care", "dialysis": "dialysis",
         "mri": "mri", "ct_scan": "ct scan", "nicu": "nicu", "cancer_treatment": "chemotherapy", "burn_care": "burn care",
         "cataract": "cataract surgery"}
TESTS = [  # (label, name pattern, city to search, location pattern, services it certainly offers)
    ("AIIMS New Delhi", r"all india institute of medical sciences|\baiims\b", "New Delhi", r"delhi",
     ["cardiac_surgery", "icu", "trauma_emergency", "dialysis", "mri", "ct_scan", "nicu", "cancer_treatment"]),
    ("Tata Memorial Hospital", r"tata memorial", "Mumbai", r"mumbai|bombay", ["cancer_treatment", "ct_scan", "mri", "icu"]),
    ("CMC Vellore", r"christian medical college|\bcmc\b", "Vellore", r"vellore",
     ["cardiac_surgery", "dialysis", "nicu", "cancer_treatment", "icu", "trauma_emergency", "mri"]),
    ("PGIMER Chandigarh", r"post ?graduate institute of medical education|\bpgimer\b|\bpgi\b", "Chandigarh", r"chandigarh",
     ["cardiac_surgery", "dialysis", "nicu", "trauma_emergency", "cancer_treatment"]),
    ("Narayana Health Bangalore", r"narayana (health|hrudayalaya|institute of cardiac)", "Bangalore", r"bangalore|bengaluru",
     ["cardiac_surgery", "icu"]),
    ("Apollo Hospitals Chennai", r"apollo hospital", "Chennai", r"chennai|madras", ["cardiac_surgery", "dialysis", "icu", "mri", "ct_scan"]),
    ("Kidwai Cancer Institute", r"kidwai", "Bangalore", r"bangalore|bengaluru", ["cancer_treatment"]),
    ("Cancer Institute (WIA) Adyar", r"cancer institute|adyar cancer", "Chennai", r"chennai|madras", ["cancer_treatment"]),
    ("Safdarjung Hospital", r"safdarjung|vardhman mahavir", "New Delhi", r"delhi", ["burn_care", "trauma_emergency"]),
    ("Aravind Eye Hospital", r"aravind eye", "Madurai", r"madurai", ["cataract"]),
    ("Sankara Nethralaya", r"sankara nethralaya", "Chennai", r"chennai|madras", ["cataract"]),
]
nm = silver["name"].fillna("").str.lower()
lc = (silver["city"].fillna("") + " " + silver["address"].fillna("") + " " + silver["pin_district"].fillna("")).str.lower()
rows = []
for label, name_re, city, city_re, services in TESTS:
    ids = set(silver.loc[nm.str.contains(name_re, regex=True) & lc.str.contains(city_re, regex=True), "unique_id"])
    for sv in services:
        keys = [sv] if sv in SERVICES else [k for k in SERVICES if sv in k]
        hit = gold[gold["unique_id"].isin(ids) & gold["service"].isin(keys)].sort_values("trust_score", ascending=False)
        level = hit["trust_level"].iloc[0] if len(hit) else ("hospital not in data" if not ids else "service not found")
        rank, n_trusted = None, None
        if ids:
            header, picks = recommend(QUERY[sv], city, top_n=500, max_km=50)
            if picks is not None:
                picks = picks.reset_index(drop=True)
                pos = picks.index[picks["unique_id"].isin(ids)]
                rank = int(pos[0]) + 1 if len(pos) else None
                n_trusted = int((picks["trust_level"] != "Claimed only").sum())
        rows.append(dict(hospital=label, service=sv, records=len(ids), level=level,
                         score=int(hit["trust_score"].iloc[0]) if len(hit) else None,
                         rank_in_city_search=rank, trusted_options_in_city=n_trusted,
                         evidence=hit["evidence"].iloc[0][0][:90] if len(hit) else ""))
famous = pd.DataFrame(rows)
pd.set_option("display.width", 250)
print(famous.drop(columns="evidence").to_string())
found = famous[~famous["level"].isin(["hospital not in data"])]
print(f"\n{len(famous)} hospital x service pairs; hospital present in the data for {len(found)}")
print("levels when the hospital is in the data:", found["level"].value_counts().to_dict())
top5 = found["rank_in_city_search"].between(1, 5).sum()
print(f"ranked in the top 5 of its own city's search: {top5} of {len(found)}")
display(famous)


/home/spark-e777f618-12d4-4805-9ddb-f3/.ipykernel/75/command-8903312773501559-1529888396:29: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ids = set(silver.loc[nm.str.contains(name_re, regex=True) & lc.str.contains(city_re, regex=True), "unique_id"])


                        hospital           service  records                 level  score  rank_in_city_search  trusted_options_in_city
0                AIIMS New Delhi   cardiac_surgery        0  hospital not in data    NaN                  NaN                      NaN
1                AIIMS New Delhi               icu        0  hospital not in data    NaN                  NaN                      NaN
2                AIIMS New Delhi  trauma_emergency        0  hospital not in data    NaN                  NaN                      NaN
3                AIIMS New Delhi          dialysis        0  hospital not in data    NaN                  NaN                      NaN
4                AIIMS New Delhi               mri        0  hospital not in data    NaN                  NaN                      NaN
5                AIIMS New Delhi           ct_scan        0  hospital not in data    NaN                  NaN                      NaN
6                AIIMS New Delhi              nicu     

hospital,service,records,level,score,rank_in_city_search,trusted_options_in_city,evidence
AIIMS New Delhi,cardiac_surgery,0,hospital not in data,null,null,null,
AIIMS New Delhi,icu,0,hospital not in data,null,null,null,
AIIMS New Delhi,trauma_emergency,0,hospital not in data,null,null,null,
AIIMS New Delhi,dialysis,0,hospital not in data,null,null,null,
AIIMS New Delhi,mri,0,hospital not in data,null,null,null,
AIIMS New Delhi,ct_scan,0,hospital not in data,null,null,null,
AIIMS New Delhi,nicu,0,hospital not in data,null,null,null,
AIIMS New Delhi,cancer_treatment,0,hospital not in data,null,null,null,
Tata Memorial Hospital,cancer_treatment,1,Verified,60.0,9.0,39.0,"procedure/capability: Outpatient oncology consultations are offered for Hematolymphoid, Pe"
Tata Memorial Hospital,ct_scan,1,Verified,65.0,37.0,73.0,equipment: CT Scan Machine - 3 units (128-slice CT scanners)


In [0]:
# Step 12: are the famous hospitals in the data at all (any city)? + compact view of Step 11
for label, name_re, city, city_re, services in TESTS:
    m_ = silver[nm.str.contains(name_re, regex=True)]
    print(f"{label[:26]:26} {len(m_):>3} records: " + "; ".join((m_["name"].str[:40] + " (" + m_["city"].fillna("?") + ")").head(4)))
print()
v = famous[famous["level"] != "hospital not in data"]
print(v[["hospital", "service", "level", "score", "rank_in_city_search", "trusted_options_in_city"]].to_string(index=False))


AIIMS New Delhi              8 records: All India Institute of Medical Sciences  (Patna); All India Institute of Medical Sciences (Bhubaneshwar); All India Institute of Medical Sciences, (Raipur); All India Institute of Medical Sciences, (Rishikesh)
Tata Memorial Hospital       1 records: Tata Memorial Hospital (Mumbai)
CMC Vellore                  3 records: Christian Medical College Ludhiana (Ludhiana); CMC HOSPITAL(PROGRESSIVE MEDICAL CARE RE (Hisar); Christian Medical College, Vellore Chitt (Chittoor)
PGIMER Chandigarh            2 records: Postgraduate Institute of Medical Educat (Bhubaneswar); PGIMER, Chandigarh (Chandigarh)
Narayana Health Bangalore    1 records: Narayana Hrudayalaya (Bengaluru)
Apollo Hospitals Chennai     8 records: Apollo Hospital Cbd Belapur (Mumbai); Apollo Hospital, Guwahati, India (Guwahati); Apollo Hospital (New Delhi); Apollo Hospital, Bhubaneswar (Bhubaneswar)


/home/spark-e777f618-12d4-4805-9ddb-f3/.ipykernel/75/command-8903312773501560-463787771:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m_ = silver[nm.str.contains(name_re, regex=True)]


Kidwai Cancer Institute      2 records: Kidwai Memorial Institute of Oncology (Bengaluru); SBL Homoeopathic Clinic, Kidwai Nagar (Kanpur)
Cancer Institute (WIA) Ady   5 records: Dr. Bhubaneswar Borooah Cancer Institute (Guwahati); Chittaranjan National Cancer Institute B (Kolkata); Delhi State Cancer Institutes (Delhi); Medico India Hospital & Reaserch Center  (Patna)
Safdarjung Hospital          2 records: Sparsh ENT Clinic Safdarjung Enclave, De (Safdarjung Enclave); Dr. Khosla's Dental Clinic - B-4/76, Saf (Delhi)
Aravind Eye Hospital         2 records: Aravind Eye Hospital (Hyderabad); Priyamvada Birla Aravind Eye Hospital (Kolkata)
Sankara Nethralaya           2 records: Sankara Nethralaya (Chennai); Kamalnayan Bajaj Sankara Nethralaya, New (New Town)

                 hospital          service             level  score  rank_in_city_search  trusted_options_in_city
   Tata Memorial Hospital cancer_treatment          Verified   60.0                  9.0                     39.0
   T

In [0]:
# Step 13: why do famous hospitals score low? look at what their records actually contain
for pat in [r"tata memorial", r"pgimer, chandigarh", r"narayana hrudayalaya", r"sankara nethralaya \(|^sankara nethralaya$"]:
    r = silver[nm.str.contains(pat, regex=True)]
    if not len(r):
        continue
    r = r.iloc[0]
    print("=" * 90)
    print(r["name"], "|", r["city"], "| type:", r["facility_type"], "| beds:", r["n_beds"], "| doctors:", r["n_doctors"], "| flags:", r["n_flags"])
    for col in ["procedure", "capability", "equipment"]:
        items = as_list(r[col])
        print(f"  {col} ({len(items)}):", "; ".join(items)[:300])
    print("  specialties:", ", ".join(as_list(r["specialties"]))[:300])
    print("  description:", str(r["description"])[:300])
    g_ = gold[gold["unique_id"] == r["unique_id"]]
    print(f"  gold rows: {len(g_)}; levels:", g_["trust_level"].value_counts().to_dict())


Tata Memorial Hospital | Mumbai | type: hospital | beds: 750.0 | doctors: 3.0 | flags: 1
  procedure (50): Outpatient oncology consultations are offered for Hematolymphoid, Pediatric, Breast, Head and Neck, Neuroncology, and Gynecology (pertaining to Radiation and Chemotherapy).; Chemotherapy, blood product infusion, and intravenous fluids are administered at the Day Care Centre for both General and Priv
  capability (50): Has Cardiology department; Has CVTS department; Has Endocrinology department; Has Gastroenterology department; Has Neurology department; Has Neurosurgery department; Has Nephrology department; Has Urology department; Has Anesthesiology department; Has Biochemistry department; Has Microbiology depart
  equipment (47): MRI - 3.0 Tesla, 1 unit; CT Scan Machine - 3 units (128-slice CT scanners); Digital Radiography Machines - 0; Computed Radiography Machine - 2; Color Doppler Machines - 4; Sonography Machines - 6; DSA Machine - 1; OPG machine; CBCT machine; Choukran’s Pl

In [0]:
# Step 14: VERIFICATION round 3 = 20 rows that are Verified ONLY because of fix H (the rule change that doubled Verified)
# "Only because of H": with the old rule they would not be Verified, i.e. fewer than 2 evidence fields,
# or the score without the two new +10 bonuses is below 60. None of the earlier 130 rows. Safe to re-run (never re-drawn).
from pyspark.sql import functions as SF, Window
T3 = "workspace.default.verify_sample_v3"
if spark.catalog.tableExists(T3):
    out3 = spark.table(T3).toPandas().sort_values("sample_id")
    print("loaded existing", T3)
else:
    g = gold[gold["trust_level"] == "Verified"].copy()
    bonus = g["score_reasons"].apply(lambda w: 10 * sum(("several separate places" in x) or ("large hospital" in x) for x in w))
    g = g[(g["n_hard_evidence"] < 2) | (g["trust_score"] - bonus < 60)]
    print(f"{len(g):,} facility x service pairs are Verified only because of fix H")
    seen = pd.concat([spark.table(f"workspace.default.{t}").select("unique_id", "service").toPandas()
                      for t in ["verify_sample_v1", "verify_sample_v2"]])
    g = g.merge(seen.assign(_seen=1), on=["unique_id", "service"], how="left")
    g = g[g["_seen"].isna()]
    g = g.sample(frac=1, random_state=2027).drop_duplicates("service")          # 1 row per service: many services
    pdf = pd.concat([g[g["service_source"] == "curated"].head(10), g[g["service_source"] == "specialty"].head(10)])
    pdf = pdf.sample(frac=1, random_state=9).reset_index(drop=True)
    src = spark.table("workspace.default.silver_facilities").select("unique_id", "source_urls").toPandas()
    pdf = pdf.merge(src, on="unique_id", how="left")
    pdf.insert(0, "sample_id", range(201, 201 + len(pdf)))
    pdf.insert(1, "reviewer", ["Joyce" if i < 211 else "Tim" for i in pdf["sample_id"]])
    ev = pdf["evidence"].apply(list)
    for j in range(3):
        pdf[f"evidence_{j + 1}"] = ev.apply(lambda e: e[j] if len(e) > j else "")
    pdf["how_scored"] = pdf["score_reasons"].apply(lambda x: "; ".join(x))
    pdf["cautions"] = pdf["problems"].apply(lambda x: "; ".join(x))
    pdf["links_to_check"] = pdf.apply(lambda r: " ".join(([r["official_website"]] if isinstance(r["official_website"], str) and r["official_website"] else [])
                                                         + list(r["source_urls"] if r["source_urls"] is not None and not isinstance(r["source_urls"], float) else [])[:3]), axis=1)
    for c in ["verdict_joyce", "reason_joyce", "web_joyce", "verdict_tim", "reason_tim", "web_tim"]:
        pdf[c] = ""
    out3 = pdf[["sample_id", "reviewer", "service_label", "service_source", "name", "facility_type", "city", "state",
                "evidence_1", "evidence_2", "evidence_3", "links_to_check",
                "verdict_joyce", "reason_joyce", "web_joyce", "verdict_tim", "reason_tim", "web_tim",
                "trust_level", "trust_score", "how_scored", "cautions", "unique_id", "service"]]
    spark.createDataFrame(out3.astype(str)).write.saveAsTable(T3)
    print("saved", T3)
print(f"{len(out3)} rows, {out3['service'].nunique()} different services | by reviewer:", out3["reviewer"].value_counts().to_dict())
print("all of these are Verified now; trust_level, trust_score, how_scored, cautions are the LAST columns: hide them while reviewing")
display(out3)


loaded existing workspace.default.verify_sample_v3
20 rows, 20 different services | by reviewer: {'Joyce': 10, 'Tim': 10}
all of these are Verified now; trust_level, trust_score, how_scored, cautions are the LAST columns: hide them while reviewing


sample_id,reviewer,service_label,service_source,name,facility_type,city,state,evidence_1,evidence_2,evidence_3,links_to_check,verdict_joyce,reason_joyce,web_joyce,verdict_tim,reason_tim,web_tim,trust_level,trust_score,how_scored,cautions,unique_id,service
201,Joyce,Dialysis,curated,MIOT International,hospital,Chennai,Tamil Nadu,procedure/capability: Dialysis,procedure/capability: MIOT Institute of Nephrology provides dialysis,specialty: nephrology,miotinternational.com https://wellnessdestinationindia.com/success-story/the-best-oral-mouth-cancer-treatment-hospitals-in-chennai https://www.dheerajbojwani.com/french-list-of-top-10-neurosurgeons-best-price-neurosurgery-hospitals-chennai.html https://vshospitals.com/best-cardiologist-in-chennai/,,,,,,,Verified,65,+30 procedure/capability mentions it (2); +10 listed as a specialty; +10 stated in several separate places; +10 large hospital (1000 beds); +5 size known (beds/doctors),,aa1bca39-e513-4582-bed8-f563805d6f07,dialysis
202,Joyce,Newborn intensive care (NICU),curated,Raiganj Government Medical College And Hospital,hospital,Raiganj,West Bengal,"procedure/capability: Specialized units: Critical Care Unit (CCU), Special New Born Care Unit (SNCU), Thalassemia Control Unit (TCU), Dialysis Unit, Blood Bank and Blood Component Separation Unit, and Link Anti-Retroviral Centre",procedure/capability: Plans to establish Newborn Intensive Care Unit (NICU) and Paediatric Intensive Care Unit (PICU),specialty: neonatology perinatal medicine,raiganjgmch.org https://uttardinajpur.gov.in/public-utility/bank-3-17/ https://uttardinajpur.gov.in/district-health-family-welfare-department/ https://pubmed.ncbi.nlm.nih.gov/36993054/,,,,,,,Verified,60,+30 procedure/capability mentions it (2); +10 listed as a specialty; +10 stated in several separate places; +10 large hospital (508 beds); +5 size known (beds/doctors),-5 coordinates disagree with PIN code,eac16a4a-5a75-4524-9abe-98e8397128d3,nicu
203,Joyce,CT scan,curated,Lourdes Hospital,hospital,Kochi,Kerala,procedure/capability: CT Scan,equipment: CT scanner (computed tomography),equipment: CT scanner,lourdeshospital.in https://en.wikipedia.org/wiki/Kochi https://www.onefivenine.com/india/Listing/Locality/hospitals/Thrissur/Thrissur/Mulayam https://www.onefivenine.com/india/Places/Taluk/Thrissur/Guruvayoor/Hospital,,,,,,,Verified,65,+30 procedure/capability mentions it (1); +25 equipment listed (2); +10 stated in several separate places; +5 size known (beds/doctors),-5 coordinates disagree with PIN code,d9cca233-cf00-40cc-9417-28da0dbf0240,ct_scan
204,Joyce,Cancer treatment (chemo / radiation),curated,Pratiksha Hospitals,hospital,Guwahati,Assam,procedure/capability: Performs chemotherapy sessions,procedure/capability: Performs around 2000 chemotherapy sessions annually,equipment: Linear Accelerator (LINAC) for external beam radiation therapy,pratikshahospitals.com https://www.makemytrip.com/hotels/ashirwad_homestay-details-guwahati.html https://corashospital.com/dr-john-mukhopadhaya/ http://www.monjoven.com/,,,,,,,Verified,65,+30 procedure/capability mentions it (2); +25 equipment listed (1); +10 listed as a specialty; +10 stated in several separate places,-10 specialty list looks inflated,17c02c25-710e-4e30-a313-82df5b9d3572,cancer_treatment
205,Joyce,Gynecologist,specialty,Sevak Maternity & Surgical Hospital,hospital,Ahmedabad,Gujarat,procedure/capability: Specializes in obstetrics and gynecology,procedure/capability: Obstetrics and gynecology services,"procedure/capability: Associated with Dr Kirtan Vyas, an obstetrician, gynecologist, infertility specialist and laparoscopic surgeon",sevakhospital.com https://www.justdial.com/Ahmedabad/Sevak-Maternity-Surgical-Hospital-Motera-Stadium-Road-Motera/079PXX79-XX79-170528120925-M4G3_BZDET https://www.justdial.com/Ahmedabad/Dr-Kirtan-M-Vyas-Opposite-Sangath-2-Bus-Stop-Motera/079PXX79-XX79-221230083033-J4A6_BZDET https://indialocaldirectory.com/place/sevak-maternity-surgical-hospital/,,,,,,,Verified,65,+30 

In [0]:
# Step 15: ROUND 3 result = are the rows that became Verified only because of fix H really correct?
# Verdicts copied from the round-3 Google Sheet (sample_id 201..220; Joyce 201-210, Tim 211-220). Reasons stay in the sheet.
import pandas as pd
V3 = "YUYUYUYYYAYYYYYYYYYU"          # Y yes, N no, U unsure, A not a service
WEB3 = "CFCFCFCCCFCCCCCCCCCF"         # C website confirms, F website found but silent, S no working website
r3 = spark.table("workspace.default.verify_sample_v3").select("sample_id", "reviewer", "unique_id", "service", "service_label",
                                                              "service_source", "trust_level", "trust_score").toPandas()
r3["sample_id"] = r3["sample_id"].astype(int)
r3 = r3.sort_values("sample_id")
assert len(V3) == len(WEB3) == len(r3) == 20
r3["verdict"] = [{"Y": "Yes", "N": "No", "U": "Unsure", "A": "N/A"}[c] for c in V3]
r3["web"] = [{"C": "confirmed", "F": "not-found", "S": "no-site"}[c] for c in WEB3]
spark.createDataFrame(r3).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.verify_results_v3")
print("ROUND 3 (20 rows Verified only because of fix H):", r3["verdict"].value_counts().to_dict())
print("by kind of service:")
print(pd.crosstab(r3["service_source"], r3["verdict"]).to_string())
print("website check:", r3["web"].value_counts().to_dict())


ROUND 3 (20 rows Verified only because of fix H): {'Yes': 15, 'Unsure': 4, 'N/A': 1}
by kind of service:
verdict         N/A  Unsure  Yes
service_source                  
curated           0       3    7
specialty         1       1    8
website check: {'confirmed': 15, 'not-found': 5}


In [0]:
# Appendix: GOLD v1 scoring (the first version). Kept for reference only: it is wrapped in a function, so running this cell does NOTHING.
# Step 3 replaced it. Its saved output lives on as workspace.default.gold_facility_services_v1 (used for the before/after comparison).
def _legacy_gold_v1_scoring():
    # (was Cell 3) score EVERY facility x service, using a word index so each service only checks relevant facilities
    import datetime
    from bisect import bisect_left
    from collections import defaultdict

    SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")
    QUANT = re.compile(r"\b\d+\s*-?\s*(bed(ded)?s?|machines?|units?|stations?|slices?|tesla|ventilators?|incubators?|warmers?|cots?|chairs?|theatres?|ots?)\b", re.I)
    EMERG = re.compile(r"24\s*[x/*]\s*7|24 ?hours?|round[- ]the[- ]clock|emergency", re.I)
    SERIOUS = ["flag_mixed_identity", "flag_state_mismatch", "flag_duplicate"]
    PENALTIES = [("flag_mixed_identity", 30, "record may merge two different places"),
                 ("flag_state_mismatch", 15, "address state conflicts with PIN code"),
                 ("flag_duplicate", 10, "duplicate name in the same city"),
                 ("flag_too_many_specialties", 10, "specialty list looks inflated"),
                 ("flag_implausible_counts", 10, "doctor/bed numbers look unrealistic"),
                 ("flag_coords_far_from_pin", 5, "coordinates disagree with PIN code")]

    # 1) prepare each facility's evidence once (original text for quoting, normalised text for matching)
    F = []
    for r in silver.to_dict("records"):
        proc = as_list(r["procedure"]) + as_list(r["capability"])
        equip = as_list(r["equipment"])
        desc = [x for x in SENT_SPLIT.split(r["description"] or "") if x.strip()]
        codes = {norm_code(x) for x in as_list(r["specialties"])} - {""}
        F.append(dict(r=r, proc=proc, equip=equip, desc=desc, codes=codes,
                      proc_n=[nz(x) for x in proc], equip_n=[nz(x) for x in equip], desc_n=[nz(x) for x in desc]))

    # 2) word index: word -> facilities whose evidence text contains it; specialty -> facilities listing it
    postings = defaultdict(set)
    for i, f in enumerate(F):
        for w in set(re.findall(r"[a-z]+", " ".join(f["proc_n"] + f["equip_n"] + f["desc_n"]))):
            postings[w].add(i)
    vocab = sorted(postings)
    code_to_fac = defaultdict(set)
    for i, f in enumerate(F):
        for c in f["codes"]:
            code_to_fac[c].add(i)

    def text_candidates(stems):
        out = None
        for s in stems:
            j, hit = bisect_left(vocab, s), set()
            while j < len(vocab) and vocab[j].startswith(s):
                hit |= postings[vocab[j]]
                j += 1
            out = hit if out is None else out & hit
            if not out:
                return set()
        return out or set()

    # 3) same scoring rules as 04_gold_scoring
    def score_pair(f, s):
        r = f["r"]
        if s["source"] == "curated":
            hit_spec = sorted(c for c in f["codes"] if s["specialty_re"].search(c.replace(" ", "")))
        else:
            hit_spec = [s["code"]] if s["code"] in f["codes"] else []
        tr, er = s["text_re"], s["equipment_re"]
        hit_proc = [o for o, n in zip(f["proc"], f["proc_n"]) if tr.search(n)]
        hit_equip = [o for o, n in zip(f["equip"], f["equip_n"]) if er.search(n) or tr.search(n)]
        hit_desc = [o for o, n in zip(f["desc"], f["desc_n"]) if tr.search(n)]
        if not (hit_spec or hit_proc or hit_equip or hit_desc):
            return None
        pts, why = 0, []
        if hit_proc:  pts += 30; why.append(f"+30 procedure/capability mentions it ({len(hit_proc)})")
        if hit_equip: pts += 25; why.append(f"+25 equipment listed ({len(hit_equip)})")
        if hit_desc:  pts += 15; why.append("+15 description mentions it")
        if hit_spec:  pts += 10; why.append("+10 listed as a specialty")
        quantified = [x for x in hit_proc + hit_equip + hit_desc if QUANT.search(x)]
        if quantified: pts += 10; why.append("+10 concrete numbers in the evidence")
        if s["emergency"] and any(EMERG.search(x) for x in f["proc"] + f["desc"]):
            pts += 5; why.append("+5 24/7 or emergency wording")
        if pd.notna(r["n_beds"]) or pd.notna(r["n_doctors"]):
            pts += 5; why.append("+5 size known (beds/doctors)")
        pen, problems, ftype = 0, [], r["facility_type"]
        type_implausible = s["hospital_level"] and ftype in ("dentist", "pharmacy")
        if type_implausible: pen += 40; problems.append(f"-40 a {ftype} claiming a hospital-level service")
        if s["hospital_level"] and ftype == "clinic" and pd.isna(r["n_beds"]):
            pen += 15; problems.append("-15 clinic with no beds listed claims a hospital-level service")
        for flag, p, txt in PENALTIES:
            if r[flag]:
                pen += p; problems.append(f"-{p} {txt}")
        score = int(max(0, min(100, pts - pen)))
        n_hard = sum(bool(h) for h in (hit_proc, hit_equip, hit_desc))
        serious = type_implausible or any(r[x] for x in SERIOUS)
        level = ("Verified" if score >= 60 and n_hard >= 2 and not serious
                 else "Likely" if score >= 35 and n_hard >= 1 else "Claimed only")
        labelled = ([("procedure/capability", x) for x in hit_proc] + [("equipment", x) for x in hit_equip]
                    + [("description", x) for x in hit_desc] + [("specialty", x) for x in hit_spec])
        labelled.sort(key=lambda fx: (not QUANT.search(fx[1]), fx[0] == "specialty"))
        return dict(unique_id=r["unique_id"], trust_score=score, trust_level=level, n_hard_evidence=n_hard,
                    ev_procedure=bool(hit_proc), ev_equipment=bool(hit_equip), ev_description=bool(hit_desc),
                    ev_specialty=bool(hit_spec), ev_quantified=bool(quantified),
                    evidence=[f"{a}: {b.strip()[:240]}" for a, b in labelled[:5]],
                    score_reasons=why, problems=problems, points_evidence=int(pts), points_penalty=int(pen))

    # 4) run all services
    t0, rows = datetime.datetime.now(), []
    for key, s in SERVICES.items():
        cand = range(len(F)) if s["source"] == "curated" else (code_to_fac[s["code"]] | text_candidates(s["stems"]))
        for i in cand:
            out = score_pair(F[i], s)
            if out:
                out.update(service=key, service_label=s["label"], service_source=s["source"])
                rows.append(out)
    scores = pd.DataFrame(rows)
    print(f"scored {len(scores):,} facility x service pairs in {(datetime.datetime.now() - t0).seconds}s")

    # 5) attach display columns, check, save
    show_cols = ["unique_id", "name", "facility_type", "operator_type", "address", "city", "state", "pincode",
                 "pin_district", "loc_lat", "loc_lon", "loc_source", "flag_location_uncertain",
                 "official_phone", "official_website", "n_beds", "n_doctors", "n_flags"]
    gold = scores.merge(silver[show_cols], on="unique_id", how="left")
    assert not gold.duplicated(["unique_id", "service"]).any()
    assert gold["evidence"].str.len().min() >= 1
    gold["silver_source"] = "workspace.default.silver_facilities"
    gold["gold_built_at"] = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    GOLD = "workspace.default.gold_facility_services"
    spark.createDataFrame(gold).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD)

    # service catalogue for the app: one row per service with counts per trust level
    cat = (gold.pivot_table(index=["service", "service_label", "service_source"], columns="trust_level",
                            values="unique_id", aggfunc="count", fill_value=0)
               .reindex(columns=["Verified", "Likely", "Claimed only"], fill_value=0).reset_index())
    cat.columns = ["service", "service_label", "service_source", "n_verified", "n_likely", "n_claimed_only"]
    cat["n_facilities"] = cat[["n_verified", "n_likely", "n_claimed_only"]].sum(axis=1)
    cat = cat.sort_values(["service_source", "n_facilities"], ascending=[True, False])
    spark.createDataFrame(cat).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.gold_services")

    print(f"saved {GOLD}: {len(gold):,} rows, {gold['service'].nunique():,} services, {gold['unique_id'].nunique():,} facilities")
    print("trust levels overall:", gold["trust_level"].value_counts().to_dict())
    print(f"saved workspace.default.gold_services: {len(cat):,} services;",
          f"{(cat['n_verified'] > 0).sum():,} have at least one Verified facility,",
          f"{((cat['n_verified'] + cat['n_likely']) > 0).sum():,} have at least one Verified or Likely")
    display(cat.sort_values("n_facilities", ascending=False).head(25))